# Imports

This section imports all required libraries for:
- File handling
- Numerical computation
- Progress tracking
- Memory diagnostics

We keep this minimal and clean to avoid unnecessary dependencies.

In [1]:
# ================================
# Imports
# ================================

import os                  # file system operations
import re                  # regex for parsing file names
import glob                # file pattern matching
import math                # math utilities
import time                # timing execution
import gc                  # garbage collection

import psutil              # system + memory monitoring
import numpy as np         # numerical operations
from tqdm import tqdm      # progress bars
from collections import defaultdict  # efficient grouping

In [2]:
import os
from multiprocessing.shared_memory import SharedMemory

_SHM_NAMES = ["boi_ids_flat", "boi_vals_flat", "boi_offsets", "boi_sig_all", "boi_x_dense"]

for _name in _SHM_NAMES:
    # Try Python API first
    try:
        _shm = SharedMemory(create=False, name=_name)
        _shm.close()
        _shm.unlink()
        print(f"Released: {_name}")
        continue
    except Exception:
        pass
    # Fallback: delete the file directly
    _path = f"/dev/shm/{_name}"
    if os.path.exists(_path):
        os.remove(_path)
        print(f"Force removed: {_name}")
    else:
        print(f"Not found (already clean): {_name}")

print("Shared memory cleanup done.")

Not found (already clean): boi_ids_flat
Not found (already clean): boi_vals_flat
Not found (already clean): boi_offsets
Not found (already clean): boi_sig_all
Not found (already clean): boi_x_dense
Shared memory cleanup done.


# Memory Monitoring Utilities

We are working with large datasets (~50K polygons), so memory usage matters.

This section provides helper functions to:
- Track RAM usage of the process
- Monitor system memory availability
- Print readable diagnostics during heavy operations

Useful during:
- Encoding loading
- Index building
- Query execution

In [3]:
# ================================
# Memory Tracking Utilities
# ================================

def get_memory_stats():
    """
    Returns current memory usage stats for:
    - This Python process
    - Overall system memory
    """
    process = psutil.Process(os.getpid())
    mem_info = process.memory_info()

    # Process-level memory
    rss_gb = mem_info.rss / (1024 ** 3)   # Resident memory (actual RAM used)
    vms_gb = mem_info.vms / (1024 ** 3)   # Virtual memory

    # System-level memory
    sys_mem = psutil.virtual_memory()
    avail_gb = sys_mem.available / (1024 ** 3)
    total_gb = sys_mem.total / (1024 ** 3)
    used_pct = sys_mem.percent

    return {
        "rss_gb": rss_gb,
        "vms_gb": vms_gb,
        "avail_gb": avail_gb,
        "total_gb": total_gb,
        "used_pct": used_pct
    }


def print_memory_stats(tag=""):
    """
    Pretty-print memory usage.
    
    Example:
        print_memory_stats("After loading encodings")
    """
    m = get_memory_stats()
    prefix = f"[{tag}] " if tag else ""

    print(
        f"{prefix}"
        f"RSS={m['rss_gb']:.2f} GB | "
        f"VMS={m['vms_gb']:.2f} GB | "
        f"Avail={m['avail_gb']:.2f}/{m['total_gb']:.2f} GB | "
        f"Used={m['used_pct']:.1f}%"
    )

# Load Weighted Encodings (Parallel)

This section loads the weighted encoding files in parallel.

Design:
- each worker parses one encoding file
- file name provides the starting polygon ID
- rows are converted into sparse in-memory form:
  - `ids`
  - `vals`
- the main process merges all parsed rows into the final `encodings` list

This replaces the earlier serial encoding-load step.

In [4]:
# ================================
# Load weighted encodings (parallel) with disk cache
# ================================

import math
import os
import re
import glob
import time
import pickle
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

# ================================
# Parameters
# ================================

# Directory containing weighted encoding files like:
#   real_0.txt, real_250.txt, ...
ENCODING_DIR = "/raid/ruban/encodings/pk-real50k0.002"

# File prefix for weighted encoding files
ENCODING_FILE_PREFIX = "real_"

# Ground truth directory
GROUND_TRUTH_DIR = "/raid/ruban/groundtruth/pk-query-50k"

_dataset_key = os.path.basename(ENCODING_DIR.rstrip("/"))
# Cache directory / file
CACHE_DIR = f"/raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/{_dataset_key}/"

os.makedirs(CACHE_DIR, exist_ok=True)
print(f"CACHE_DIR: {CACHE_DIR}")

ENCODINGS_CACHE_PATH = os.path.join(CACHE_DIR, "encodings_payload.pkl")

# Total number of polygons in this dataset
def count_polygons(encoding_dir, file_prefix="real_"):
    pattern = os.path.join(encoding_dir, f"{file_prefix}*.txt")
    files = glob.glob(pattern)
    total = 0
    for fpath in sorted(files):
        with open(fpath, "r") as f:
            total += sum(1 for line in f if line.strip())
    return total

POLY_COUNT = count_polygons(ENCODING_DIR, ENCODING_FILE_PREFIX)
print(f"Auto-detected POLY_COUNT: {POLY_COUNT:,}")

# Split:
# first 80% -> database
# last 20%  -> query set
TRAIN_RATIO = 0.80

# Use full query split for now
MAX_QUERIES = None

# Values with abs(value) <= ZERO_TOL are treated as zero
ZERO_TOL = 1e-12

# In-memory dtypes
ID_DTYPE = np.int32
VAL_DTYPE = np.float32

# Recall points to evaluate later
RECALL_KS = [10, 50, 500]

# Verbose diagnostics
VERBOSE = True

# Derived split indices
DATA_END = math.ceil(POLY_COUNT * TRAIN_RATIO)
QUERY_START = DATA_END
QUERY_END = POLY_COUNT if MAX_QUERIES is None else min(POLY_COUNT, QUERY_START + MAX_QUERIES)

print("=== Parameter Summary ===")
print(f"ENCODING_DIR           : {ENCODING_DIR}")
print(f"ENCODING_FILE_PREFIX   : {ENCODING_FILE_PREFIX}")
print(f"GROUND_TRUTH_DIR       : {GROUND_TRUTH_DIR}")
print(f"POLY_COUNT             : {POLY_COUNT:,}")
print(f"TRAIN_RATIO            : {TRAIN_RATIO}")
print(f"DATA_END               : {DATA_END:,}")
print(f"QUERY_START            : {QUERY_START:,}")
print(f"QUERY_END              : {QUERY_END:,}")
print(f"NUM_QUERIES            : {QUERY_END - QUERY_START:,}")
print(f"ZERO_TOL               : {ZERO_TOL}")
print(f"ID_DTYPE               : {ID_DTYPE}")
print(f"VAL_DTYPE              : {VAL_DTYPE}")
print(f"RECALL_KS              : {RECALL_KS}")
print(f"CACHE_DIR              : {CACHE_DIR}")
print(f"ENCODINGS_CACHE_PATH   : {ENCODINGS_CACHE_PATH}")

print_memory_stats("After parameters")

# Parallel settings
ENC_LOAD_WORKERS = 16

print("=== Parallel Encoding Load Parameters ===")
print(f"ENC_LOAD_WORKERS       : {ENC_LOAD_WORKERS}")
print(f"ENCODING_DIR           : {ENCODING_DIR}")
print(f"ENCODING_FILE_PREFIX   : {ENCODING_FILE_PREFIX}")


def _parse_weighted_line_auto_worker(line, zero_tol=1e-12):
    """
    Worker-safe line parser.

    Supported formats:
    1) Dense floats
    2) Sparse idx:value
    3) Sparse alternating idx value pairs
    """
    line = line.strip()

    if not line:
        return (
            np.empty(0, dtype=ID_DTYPE),
            np.empty(0, dtype=VAL_DTYPE)
        )

    toks = line.split()

    # Sparse idx:value
    if ":" in toks[0]:
        ids = []
        vals = []

        for tok in toks:
            cell_str, val_str = tok.split(":", 1)
            v = float(val_str)
            if abs(v) > zero_tol:
                ids.append(int(cell_str))
                vals.append(v)

        return (
            np.asarray(ids, dtype=ID_DTYPE),
            np.asarray(vals, dtype=VAL_DTYPE)
        )

    # Sparse alternating idx value
    is_even = (len(toks) % 2 == 0)

    def _looks_like_int_token(s):
        return re.fullmatch(r"[+-]?\d+", s) is not None

    if is_even and all(_looks_like_int_token(toks[i]) for i in range(0, len(toks), 2)):
        ids = []
        vals = []

        for i in range(0, len(toks), 2):
            cell_id = int(toks[i])
            v = float(toks[i + 1])

            if abs(v) > zero_tol:
                ids.append(cell_id)
                vals.append(v)

        return (
            np.asarray(ids, dtype=ID_DTYPE),
            np.asarray(vals, dtype=VAL_DTYPE)
        )

    # Dense float row
    dense = np.fromstring(line, sep=" ", dtype=VAL_DTYPE)

    if dense.size == 0:
        return (
            np.empty(0, dtype=ID_DTYPE),
            np.empty(0, dtype=VAL_DTYPE)
        )

    nz = np.flatnonzero(np.abs(dense) > zero_tol).astype(ID_DTYPE)
    vals = dense[nz].astype(VAL_DTYPE, copy=False)

    return nz, vals


def _extract_file_start_id(fpath, file_prefix):
    base = os.path.basename(fpath)
    m = re.search(rf"{re.escape(file_prefix)}(\d+)\.txt$", base)
    if not m:
        raise ValueError(f"Unexpected encoding file name format: {base}")
    return int(m.group(1))


def _parse_one_encoding_file(args):
    """
    Worker function for one encoding file.

    Returns
    -------
    rows : list of (poly_id, ids, vals)
    """
    fpath, file_prefix, start_id, end_id, zero_tol = args

    file_start_id = _extract_file_start_id(fpath, file_prefix)
    rows = []

    with open(fpath, "r") as f:
        for local_row_idx, line in enumerate(f):
            poly_id = file_start_id + local_row_idx

            if poly_id < start_id or poly_id >= end_id:
                continue

            ids, vals = _parse_weighted_line_auto_worker(line, zero_tol=zero_tol)
            rows.append((poly_id, ids, vals))

    return rows


def load_weighted_encodings_parallel(encoding_dir, start_id=0, end_id=None,
                                     file_prefix="real_", zero_tol=1e-12,
                                     num_workers=16, show_progress=True):
    """
    Parallel weighted encoding loader.

    Returns
    -------
    encodings : list[(ids, vals)]
    """
    if end_id is None:
        end_id = POLY_COUNT

    pattern = os.path.join(encoding_dir, f"{file_prefix}*.txt")
    files = glob.glob(pattern)

    if not files:
        raise FileNotFoundError(f"No files found matching pattern: {pattern}")

    files = sorted(files, key=lambda f: _extract_file_start_id(f, file_prefix))

    n_rows = end_id - start_id
    empty_ids = np.empty(0, dtype=ID_DTYPE)
    empty_vals = np.empty(0, dtype=VAL_DTYPE)
    encodings = [(empty_ids, empty_vals) for _ in range(n_rows)]

    jobs = [
        (fpath, file_prefix, start_id, end_id, zero_tol)
        for fpath in files
    ]

    ctx = mp.get_context("fork")

    loaded_rows = 0
    nonempty_rows = 0

    with ProcessPoolExecutor(max_workers=num_workers, mp_context=ctx) as ex:
        futures = [ex.submit(_parse_one_encoding_file, job) for job in jobs]

        iterator = as_completed(futures)
        if show_progress:
            iterator = tqdm(iterator, total=len(futures), desc="Loading weighted encodings (parallel)", unit="file")

        for fut in iterator:
            rows = fut.result()

            for poly_id, ids, vals in rows:
                encodings[poly_id - start_id] = (ids, vals)
                loaded_rows += 1
                if ids.size > 0:
                    nonempty_rows += 1

    print(f"Loaded rows             : {loaded_rows:,}")
    print(f"Non-empty rows          : {nonempty_rows:,}")
    print(f"Requested poly ID range : [{start_id}, {end_id})")

    return encodings


# ================================
# Cache-aware load
# ================================
if os.path.exists(ENCODINGS_CACHE_PATH):   
    print(f"[Cache hit] Loading encodings from disk: {ENCODINGS_CACHE_PATH}")
    t0 = time.time()

    with open(ENCODINGS_CACHE_PATH, "rb") as f:
        enc_payload = pickle.load(f)

    encodings = enc_payload["encodings"]

    # restore if present
    if "DATA_END" in enc_payload:
        DATA_END = enc_payload["DATA_END"]
    if "QUERY_START" in enc_payload:
        QUERY_START = enc_payload["QUERY_START"]
    if "QUERY_END" in enc_payload:
        QUERY_END = enc_payload["QUERY_END"]
    if "POLY_COUNT" in enc_payload:
        POLY_COUNT = enc_payload["POLY_COUNT"]

    t_load_enc = time.time() - t0
    print(f"Loaded cached encodings in {t_load_enc:.2f} sec")

else:
    print("[Cache miss] Loading weighted encodings fresh...")
    t0 = time.time()

    encodings = load_weighted_encodings_parallel(
        encoding_dir=ENCODING_DIR,
        start_id=0,
        end_id=POLY_COUNT,
        file_prefix=ENCODING_FILE_PREFIX,
        zero_tol=ZERO_TOL,
        num_workers=ENC_LOAD_WORKERS,
        show_progress=True
    )

    t_load_enc = time.time() - t0
    print(f"\nFresh parallel encoding load time: {t_load_enc:.2f} sec ({t_load_enc / 60:.2f} min)")

    print(f"[Cache save] Writing encodings to: {ENCODINGS_CACHE_PATH}")
    with open(ENCODINGS_CACHE_PATH, "wb") as f:
        pickle.dump({
            "encodings": encodings,
            "POLY_COUNT": POLY_COUNT,
            "DATA_END": DATA_END,
            "QUERY_START": QUERY_START,
            "QUERY_END": QUERY_END,
            "ZERO_TOL": ZERO_TOL,
            "ID_DTYPE": str(ID_DTYPE),
            "VAL_DTYPE": str(VAL_DTYPE),
        }, f, protocol=pickle.HIGHEST_PROTOCOL)

print_memory_stats("After encoding load")

# ================================
# Basic weighted encoding diagnostics
# ================================
nnz_counts = np.array([ids.size for ids, vals in encodings], dtype=np.int32)

nonempty_mask = (nnz_counts > 0)
num_nonempty = int(nonempty_mask.sum())
num_empty = int((~nonempty_mask).sum())

max_cell_id = -1
min_nonzero_val = None
max_nonzero_val = None

for ids, vals in encodings:
    if ids.size > 0:
        local_max_id = int(ids.max())
        if local_max_id > max_cell_id:
            max_cell_id = local_max_id

    if vals.size > 0:
        local_min_val = float(vals.min())
        local_max_val = float(vals.max())

        if min_nonzero_val is None or local_min_val < min_nonzero_val:
            min_nonzero_val = local_min_val

        if max_nonzero_val is None or local_max_val > max_nonzero_val:
            max_nonzero_val = local_max_val

print("\n=== Weighted Encoding Diagnostics ===")
print(f"Total polygons           : {len(encodings):,}")
print(f"Non-empty polygons       : {num_nonempty:,}")
print(f"Empty polygons           : {num_empty:,}")
print(f"Mean nnz / polygon       : {nnz_counts.mean():.2f}")
print(f"Median nnz / polygon     : {np.median(nnz_counts):.2f}")
print(f"p90 nnz / polygon        : {np.percentile(nnz_counts, 90):.2f}")
print(f"p99 nnz / polygon        : {np.percentile(nnz_counts, 99):.2f}")
print(f"Max nnz / polygon        : {nnz_counts.max()}")
print(f"Max cell ID seen         : {max_cell_id}")
print(f"Min nonzero value        : {min_nonzero_val}")
print(f"Max nonzero value        : {max_nonzero_val}")

CACHE_DIR: /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/pk-real50k0.002/
Auto-detected POLY_COUNT: 50,000
=== Parameter Summary ===
ENCODING_DIR           : /raid/ruban/encodings/pk-real50k0.002
ENCODING_FILE_PREFIX   : real_
GROUND_TRUTH_DIR       : /raid/ruban/groundtruth/pk-query-50k
POLY_COUNT             : 50,000
TRAIN_RATIO            : 0.8
DATA_END               : 40,000
QUERY_START            : 40,000
QUERY_END              : 50,000
NUM_QUERIES            : 10,000
ZERO_TOL               : 1e-12
ID_DTYPE               : <class 'numpy.int32'>
VAL_DTYPE              : <class 'numpy.float32'>
RECALL_KS              : [10, 50, 500]
CACHE_DIR              : /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/pk-real50k0.002/
ENCODINGS_CACHE_PATH   : /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/pk-real50k0.002/encodings_payload.pkl
[After parameters] RSS=0.08 GB | VMS=3.30 GB | Avail=1962.38/2015.68 GB | Used=2.6%
=== Parallel Encoding Load Parame

/tmp/ipykernel_3232229/1922442022.py:279: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  enc_payload = pickle.load(f)


Loaded cached encodings in 1.50 sec
[After encoding load] RSS=1.98 GB | VMS=5.20 GB | Avail=1960.47/2015.68 GB | Used=2.7%

=== Weighted Encoding Diagnostics ===
Total polygons           : 50,000
Non-empty polygons       : 49,995
Empty polygons           : 5
Mean nnz / polygon       : 4889.49
Median nnz / polygon     : 3958.00
p90 nnz / polygon        : 10535.10
p99 nnz / polygon        : 15885.02
Max nnz / polygon        : 18380
Max cell ID seen         : 18380
Min nonzero value        : 1.0000080191002736e-12
Max nonzero value        : 0.16067412495613098


# Ground Truth Loader

This section loads the existing polygon-shape ground truth.

Format assumed from the current notebook:
- one query per line
- each line is:
  `query_id, neighbor1, neighbor2, neighbor3, ...`

This is the same exact-polygon Jaccard based ground truth used in the earlier pipeline.

We will use it only for evaluation and diagnostics.

In [5]:
# ================================
# Ground truth loader with disk cache
# ================================

import os
import time
import pickle
import numpy as np

GT_CACHE_PATH = os.path.join(CACHE_DIR, "gt.pkl")

def read_ground_truth(path):
    """
    Load ground truth files and return a CSR-style structure.

    Returns
    -------
    gt_neighbors : np.ndarray[int32]   — flat array of all neighbor IDs
    gt_offsets   : np.ndarray[int64]   — gt_offsets[i]..gt_offsets[i+1] = neighbors of query i
    gt_qids      : np.ndarray[int32]   — sorted query IDs
    gt_qid_map   : dict[int, int]      — maps query ID -> row index into gt_offsets
    """
    raw = {}
    files = sorted(os.listdir(path))

    for fname in files:
        fpath = os.path.join(path, fname)
        if not os.path.isfile(fpath):
            continue
        with open(fpath, "r") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split(", ")
                if len(parts) >= 2:
                    qid = int(parts[0])
                    neighbors = [int(x) for x in parts[1:] if x.strip()]
                    raw[qid] = neighbors

    # Build CSR layout
    gt_qids = np.array(sorted(raw.keys()), dtype=np.int32)
    gt_offsets = np.zeros(len(gt_qids) + 1, dtype=np.int64)

    for i, qid in enumerate(gt_qids):
        gt_offsets[i + 1] = gt_offsets[i] + len(raw[qid])

    total = int(gt_offsets[-1])
    gt_neighbors = np.empty(total, dtype=np.int32)

    for i, qid in enumerate(gt_qids):
        s, e = gt_offsets[i], gt_offsets[i + 1]
        gt_neighbors[s:e] = raw[qid]

    gt_qid_map = {int(qid): i for i, qid in enumerate(gt_qids)}

    print(f"Ground truth loaded: {len(gt_qids):,} queries | {total:,} total neighbors")
    return gt_neighbors, gt_offsets, gt_qids, gt_qid_map


# -------------------------
# Load GT (cache-aware)
# -------------------------

print("[2] Loading ground truth...")
t0 = time.time()

GT_CSR_CACHE_PATH = os.path.join(CACHE_DIR, "gt_csr.pkl")
if os.path.exists(GT_CSR_CACHE_PATH):
    print(f"[Cache hit] Loading GT CSR from disk: {GT_CSR_CACHE_PATH}")
    with open(GT_CSR_CACHE_PATH, "rb") as f:
        _gt_payload = pickle.load(f)
    gt_neighbors = _gt_payload["gt_neighbors"]
    gt_offsets   = _gt_payload["gt_offsets"]
    gt_qids      = _gt_payload["gt_qids"]
    gt_qid_map   = _gt_payload["gt_qid_map"]
else:
    print(f"[Cache miss] Loading GT fresh from: {GROUND_TRUTH_DIR}")
    gt_neighbors, gt_offsets, gt_qids, gt_qid_map = read_ground_truth(GROUND_TRUTH_DIR)

    print(f"[Cache save] Writing GT CSR to: {GT_CSR_CACHE_PATH}")
    with open(GT_CSR_CACHE_PATH, "wb") as f:
        pickle.dump({
            "gt_neighbors": gt_neighbors,
            "gt_offsets":   gt_offsets,
            "gt_qids":      gt_qids,
            "gt_qid_map":   gt_qid_map,
        }, f, protocol=pickle.HIGHEST_PROTOCOL)

t_gt = time.time() - t0

print(f"Ground truth load time: {t_gt:.2f} sec")
print_memory_stats("After loading ground truth")

# -------------------------
# GT diagnostics
# -------------------------

gt_lengths = (gt_offsets[1:] - gt_offsets[:-1]).astype(np.int32)

print("\n=== Ground Truth Diagnostics ===")
print(f"GT query count            : {len(gt_qids):,}")
print(f"Min GT query ID           : {gt_qids[0]}")
print(f"Max GT query ID           : {gt_qids[-1]}")
print(f"Mean GT list length       : {gt_lengths.mean():.2f}")
print(f"Median GT list length     : {np.median(gt_lengths):.2f}")
print(f"Min GT list length        : {gt_lengths.min()}")
print(f"Max GT list length        : {gt_lengths.max()}")
print(f"p90 GT list length        : {np.percentile(gt_lengths, 90):.2f}")

query_ids = list(range(QUERY_START, QUERY_END))
valid_gt_query_ids = [qid for qid in query_ids if qid in gt_qid_map]

print("\n=== Query Split vs GT ===")
print(f"Query split size          : {len(query_ids):,}")
print(f"Queries with GT           : {len(valid_gt_query_ids):,}")
print(f"Queries without GT        : {len(query_ids) - len(valid_gt_query_ids):,}")

[2] Loading ground truth...
[Cache hit] Loading GT CSR from disk: /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/pk-real50k0.002/gt_csr.pkl
Ground truth load time: 0.02 sec
[After loading ground truth] RSS=2.02 GB | VMS=5.24 GB | Avail=1960.44/2015.68 GB | Used=2.7%

=== Ground Truth Diagnostics ===
GT query count            : 9,417
Min GT query ID           : 40000
Max GT query ID           : 49999
Mean GT list length       : 1065.93
Median GT list length     : 735.00
Min GT list length        : 1
Max GT list length        : 3676
p90 GT list length        : 2683.40

=== Query Split vs GT ===
Query split size          : 10,000
Queries with GT           : 9,417
Queries without GT        : 583


/tmp/ipykernel_3232229/1598027692.py:72: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  _gt_payload = pickle.load(f)


# Weighted MinHash Signatures

Now that weighted Jaccard is verified, we replace standard MinHash with weighted MinHash.

Goal:
- create compact signatures for weighted vectors
- preserve weighted Jaccard similarity approximately
- keep the rest of the banding pipeline structurally similar to the earlier notebook

Implementation choice:
- use Consistent Weighted Sampling (CWS), specifically the improved weighted MinHash method
- each signature position produces one integer-like bucket value
- the final signature matrix will later be used for banding

Important:
- this is only the signature-generation stage
- we are not building buckets or indexes yet

In [6]:
# ================================
# Weighted MinHash setup
# ================================

# Signature length
# Keep this aligned with your earlier pipeline for easier comparison
L = 100

# Random seed for reproducibility
WMH_SEED = 42

print("=== Weighted MinHash Parameters ===")
print(f"L         : {L}")
print(f"WMH_SEED  : {WMH_SEED}")


def build_weighted_minhash_params(L, seed=42):
    """
    Build random parameters for Improved Consistent Weighted Sampling (ICWS-style).

    For each signature dimension j, and each active feature i, we use:
      r_ij ~ Gamma(2,1)
      c_ij ~ Gamma(2,1)
      beta_ij ~ Uniform(0,1)

    In practice, we generate these lazily per active coordinate by indexing
    pre-generated matrices of size [L, D], where D is the max cell id + 1.

    Since D is known only after loading the data, we build these params after
    encoding diagnostics.

    Returns
    -------
    params : dict
        Contains random parameter matrices needed for weighted MinHash
    """
    D = max(int(ids.max()) for ids, vals in encodings if ids.size > 0) + 1
    rng = np.random.default_rng(seed)

    # Gamma(shape=2, scale=1) and Uniform(0,1)
    r = rng.gamma(shape=2.0, scale=1.0, size=(L, D)).astype(np.float32)
    c = rng.gamma(shape=2.0, scale=1.0, size=(L, D)).astype(np.float32)
    beta = rng.random(size=(L, D), dtype=np.float32)

    return {
        "L": L,
        "D": D,
        "r": r,
        "c": c,
        "beta": beta
    }


def weighted_minhash_signature_one(ids, vals, params):
    """
    Compute one weighted MinHash signature for a sparse nonnegative vector.

    Parameters
    ----------
    ids : np.ndarray[int]
        Active feature IDs
    vals : np.ndarray[float]
        Positive weights for those features
    params : dict
        Random parameters produced by build_weighted_minhash_params()

    Returns
    -------
    sig : np.ndarray[int64] of shape (L,)
        Weighted MinHash signature
    """
    L = params["L"]
    r = params["r"]
    c = params["c"]
    beta = params["beta"]

    # Empty vector: assign a sentinel signature
    if ids.size == 0:
        return np.full(L, -1, dtype=np.int64)

    # Safety: weighted MinHash requires positive weights
    positive_mask = vals > 0
    ids = ids[positive_mask]
    vals = vals[positive_mask]

    if ids.size == 0:
        return np.full(L, -1, dtype=np.int64)

    log_vals = np.log(vals.astype(np.float32))

    sig = np.empty(L, dtype=np.int64)

    # For each signature dimension, choose the feature with minimum a_z
    for j in range(L):
        r_j = r[j, ids]
        c_j = c[j, ids]
        beta_j = beta[j, ids]

        # ICWS-style transform
        t = np.floor((log_vals / r_j) + beta_j)
        y = np.exp(r_j * (t - beta_j))
        a = c_j / (y * np.exp(r_j))

        # Winner feature index within current active set
        k = int(np.argmin(a))

        # Encode both feature id and discretized t into one hashable integer
        # This preserves more information than feature id alone.
        feature_id = int(ids[k])
        t_val = int(t[k])

        sig[j] = np.int64(feature_id) * np.int64(2147483647) + np.int64(t_val)

    return sig


print("Building weighted MinHash random parameters...")
t0 = time.time()
wmh_params = build_weighted_minhash_params(L=L, seed=WMH_SEED)
t_build_wmh = time.time() - t0

print(f"Weighted MinHash parameter build time: {t_build_wmh:.2f} sec")
print(f"Feature dimension D                  : {wmh_params['D']:,}")
print_memory_stats("After WMH parameter build")

=== Weighted MinHash Parameters ===
L         : 100
WMH_SEED  : 42
Building weighted MinHash random parameters...
Weighted MinHash parameter build time: 0.27 sec
Feature dimension D                  : 18,381
[After WMH parameter build] RSS=2.05 GB | VMS=5.27 GB | Avail=1960.41/2015.68 GB | Used=2.7%


# Generate Weighted MinHash Signatures (Parallel)

This section computes weighted MinHash signatures in parallel.

Design:
- split the polygon encodings into chunks
- use `ProcessPoolExecutor`
- each worker computes signatures for one chunk
- merge chunk results into the final signature matrix

This replaces the earlier serial signature-build step.

In [7]:
# ================================
# Generate weighted MinHash signatures (parallel) with disk cache
# ================================

import os
import time
import pickle
import numpy as np
from numba import njit, prange

# ------------------------------------------------------------
# Cache paths
# ------------------------------------------------------------
# Assumes CACHE_DIR already exists
FLAT_ENCODING_CACHE_PATH = os.path.join(CACHE_DIR, "flat_encoding_arrays.pkl")
WMH_PARAMS_CACHE_PATH = os.path.join(CACHE_DIR, f"wmh_params_L{L}.pkl")
SIG_ALL_CACHE_PATH = os.path.join(CACHE_DIR, f"sig_all_L{L}.npy")

print("=== WMH Cache Paths ===")
print(f"FLAT_ENCODING_CACHE_PATH : {FLAT_ENCODING_CACHE_PATH}")
print(f"WMH_PARAMS_CACHE_PATH    : {WMH_PARAMS_CACHE_PATH}")
print(f"SIG_ALL_CACHE_PATH       : {SIG_ALL_CACHE_PATH}")

# "hnsw"     → existing nmslib WeightedJaccard HNSW (baseline)
# "nested_lsh" → second-level MinHash banding inside large buckets
# "ivf_flat"   → FAISS IVFFlat on 100-dim signatures
# "rp_lsh"     → Random Projection LSH on signatures (our main focus)
INDEX_TYPE = "ivf_flat"

@njit(cache=True, parallel=True)
def _wmh_sig_all_numba(ids_list_flat, vals_list_flat, offsets, r, c, beta, L, N):
    """
    Compute all N signatures in parallel using Numba prange.
    ids_list_flat / vals_list_flat are 1D concatenations of all sparse vectors.
    offsets[i] = start index of polygon i in the flat arrays.
    offsets[N] = total length (sentinel).
    """
    sig_all = np.full((N, L), -1, dtype=np.int64)

    for poly_idx in prange(N):
        start = offsets[poly_idx]
        end   = offsets[poly_idx + 1]

        if start == end:
            continue

        ids  = ids_list_flat[start:end]
        vals = vals_list_flat[start:end].copy()

        n = 0
        for ki in range(ids.shape[0]):
            if vals[ki] > 0.0:
                vals[n] = np.log(vals[ki])
                ids[n]  = ids[ki]
                n += 1

        if n == 0:
            continue

        ids  = ids[:n]
        vals = vals[:n]

        for j in range(L):
            best_a          = np.inf
            best_feature_id = np.int64(-1)
            best_t          = np.int64(0)

            for ki in range(n):
                idx    = np.int64(ids[ki])
                r_j    = r[j, idx]
                c_j    = c[j, idx]
                beta_j = beta[j, idx]
                lv     = vals[ki]

                t = np.floor(lv / r_j + beta_j)
                y = np.exp(r_j * (t - beta_j))
                a = c_j / (y * np.exp(r_j))

                if a < best_a:
                    best_a          = a
                    best_feature_id = idx
                    best_t          = np.int64(t)

            sig_all[poly_idx, j] = best_feature_id * np.int64(2147483647) + best_t

    return sig_all


# ------------------------------------------------------------
# 1) Flat encoding arrays (cache-aware)
# ------------------------------------------------------------
if os.path.exists(FLAT_ENCODING_CACHE_PATH):
    print(f"[Cache hit] Loading flat encoding arrays from: {FLAT_ENCODING_CACHE_PATH}")
    t_flat0 = time.time()

    with open(FLAT_ENCODING_CACHE_PATH, "rb") as f:
        flat_payload = pickle.load(f)

    ids_flat = flat_payload["ids_flat"]
    vals_flat = flat_payload["vals_flat"]
    offsets = flat_payload["offsets"]
    total_nnz = flat_payload["total_nnz"]

    # Safety check — ensure correct dtypes
    assert ids_flat.dtype == np.int32, f"ids_flat dtype mismatch: expected int32, got {ids_flat.dtype}"
    assert vals_flat.dtype == np.float32, f"vals_flat dtype mismatch: expected float32, got {vals_flat.dtype}"
    print(f"ids_flat dtype  : {ids_flat.dtype} ✓")
    print(f"vals_flat dtype : {vals_flat.dtype} ✓")

    print(f"Loaded flat arrays in {time.time() - t_flat0:.2f} sec | total nnz: {total_nnz:,}")

else:
    print("[Cache miss] Building flat encoding arrays for Numba...")
    t_flat0 = time.time()

    offsets = np.zeros(len(encodings) + 1, dtype=np.int64)
    total_nnz = 0

    for i, (ids, vals) in enumerate(encodings):
        total_nnz += ids.size
        offsets[i + 1] = total_nnz

    ids_flat = np.empty(total_nnz, dtype=np.int32)
    vals_flat = np.empty(total_nnz, dtype=np.float32)

    for i, (ids, vals) in enumerate(encodings):
        s, e = offsets[i], offsets[i + 1]
        ids_flat[s:e] = ids
        vals_flat[s:e] = vals

    print(f"Flat array build: {time.time() - t_flat0:.2f} sec | total nnz: {total_nnz:,}")

    print(f"[Cache save] Writing flat encoding arrays to: {FLAT_ENCODING_CACHE_PATH}")
    with open(FLAT_ENCODING_CACHE_PATH, "wb") as f:
        pickle.dump({
            "ids_flat": ids_flat,
            "vals_flat": vals_flat,
            "offsets": offsets,
            "total_nnz": total_nnz,
        }, f, protocol=pickle.HIGHEST_PROTOCOL)


# ------------------------------------------------------------
# 2) WMH params for current L (cache-aware)
# ------------------------------------------------------------
if os.path.exists(WMH_PARAMS_CACHE_PATH):
    print(f"[Cache hit] Loading WMH params from: {WMH_PARAMS_CACHE_PATH}")
    t_params0 = time.time()

    with open(WMH_PARAMS_CACHE_PATH, "rb") as f:
        wmh_params = pickle.load(f)

    print(f"Loaded WMH params in {time.time() - t_params0:.2f} sec")

else:
    print(f"[Cache miss] Building WMH params for L={L} ...")
    t_params0 = time.time()

    wmh_params = build_weighted_minhash_params(L, seed=WMH_SEED)

    print(f"WMH params build time: {time.time() - t_params0:.2f} sec")

    print(f"[Cache save] Writing WMH params to: {WMH_PARAMS_CACHE_PATH}")
    with open(WMH_PARAMS_CACHE_PATH, "wb") as f:
        pickle.dump(wmh_params, f, protocol=pickle.HIGHEST_PROTOCOL)


# ------------------------------------------------------------
# 3) Numba warmup (only if needed)
# ------------------------------------------------------------
print("Numba WMH warmup...")
_ = _wmh_sig_all_numba(
    ids_flat[:offsets[1]],
    vals_flat[:offsets[1]],
    np.array([0, offsets[1]], dtype=np.int64),
    wmh_params["r"], wmh_params["c"], wmh_params["beta"],
    L, 1
)
print("Numba WMH signature warmup done — compiled and ready.")


# ------------------------------------------------------------
# 4) Signature matrix for current L (cache-aware)
# ------------------------------------------------------------
if os.path.exists(SIG_ALL_CACHE_PATH):
    print(f"[Cache hit] Loading weighted MinHash signatures from: {SIG_ALL_CACHE_PATH}")
    t0 = time.time()

    sig_all = np.load(SIG_ALL_CACHE_PATH)
    print(f"sig_all raw dtype: {sig_all.dtype}")
    # Keep int64 for nested LSH exact matching; cast to float32 only for HNSW
    if INDEX_TYPE == "hnsw":
        sig_all = sig_all.astype(np.float32)

    t_sig = time.time() - t0
    print(f"Loaded cached signatures in {t_sig:.2f} sec")

else:
    print("[Cache miss] Computing weighted MinHash signatures...")
    t0 = time.time()

    sig_all = _wmh_sig_all_numba(
        ids_flat, vals_flat, offsets,
        wmh_params["r"], wmh_params["c"], wmh_params["beta"],
        L, len(encodings)
    )

    t_sig = time.time() - t0
    print(f"\nNumba parallel weighted MinHash signature time: {t_sig:.2f} sec ({t_sig/60:.2f} min)")

    print(f"[Cache save] Writing signatures to: {SIG_ALL_CACHE_PATH}")
    np.save(SIG_ALL_CACHE_PATH, sig_all.astype(np.float32))


print(f"Signature matrix shape                        : {sig_all.shape}")
print(f"Signature dtype                               : {sig_all.dtype}")


print_memory_stats("After weighted MinHash signatures")

=== WMH Cache Paths ===
FLAT_ENCODING_CACHE_PATH : /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/pk-real50k0.002/flat_encoding_arrays.pkl
WMH_PARAMS_CACHE_PATH    : /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/pk-real50k0.002/wmh_params_L100.pkl
SIG_ALL_CACHE_PATH       : /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/pk-real50k0.002/sig_all_L100.npy
[Cache hit] Loading flat encoding arrays from: /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/pk-real50k0.002/flat_encoding_arrays.pkl


/tmp/ipykernel_3232229/1331828643.py:97: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  flat_payload = pickle.load(f)


ids_flat dtype  : int32 ✓
vals_flat dtype : float32 ✓
Loaded flat arrays in 0.92 sec | total nnz: 244,474,740
[Cache hit] Loading WMH params from: /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/pk-real50k0.002/wmh_params_L100.pkl
Loaded WMH params in 0.01 sec
Numba WMH warmup...


/tmp/ipykernel_3232229/1331828643.py:151: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  wmh_params = pickle.load(f)


Numba WMH signature warmup done — compiled and ready.
[Cache hit] Loading weighted MinHash signatures from: /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/pk-real50k0.002/sig_all_L100.npy
sig_all raw dtype: float32
Loaded cached signatures in 0.01 sec
Signature matrix shape                        : (50000, 100)
Signature dtype                               : float32
[After weighted MinHash signatures] RSS=4.00 GB | VMS=27.74 GB | Avail=1958.53/2015.68 GB | Used=2.8%


In [8]:
# ================================
# Shared Memory Setup
# All large numpy arrays placed into OS shared memory.
# Workers attach by name — zero copies, no CoW.
# ================================

from multiprocessing.shared_memory import SharedMemory

_SHM_REGISTRY = {}   # name -> SharedMemory object (keep alive in main process)

def ndarray_to_shm(arr, name):
    """
    Copy a numpy array into a named shared memory block.
    Returns the SharedMemory object (must stay alive).
    """
    shm = SharedMemory(create=True, size=arr.nbytes, name=name)
    shared = np.ndarray(arr.shape, dtype=arr.dtype, buffer=shm.buf)
    shared[:] = arr
    return shm

def shm_to_ndarray(name, shape, dtype):
    """
    Attach to an existing shared memory block and return a numpy view.
    Does NOT own the block — caller must not call shm.unlink().
    """
    shm = SharedMemory(create=False, name=name)
    arr = np.ndarray(shape, dtype=dtype, buffer=shm.buf)
    return arr, shm

def cleanup_shm_registry():
    for name, shm in _SHM_REGISTRY.items():
        try:
            shm.close()
            shm.unlink()
        except Exception:
            pass
    _SHM_REGISTRY.clear()
    print("Shared memory blocks released.")

# -----------------------------------------------
# Place arrays into shared memory
# -----------------------------------------------
print("Moving arrays to shared memory...")
t0 = time.time()

# Clean up any leftover blocks from a previous run
for _name in ["boi_ids_flat", "boi_vals_flat", "boi_offsets", "boi_sig_all"]:
    try:
        _s = SharedMemory(create=False, name=_name)
        _s.close()
        _s.unlink()
    except Exception:
        pass

_SHM_REGISTRY["boi_ids_flat"]  = ndarray_to_shm(ids_flat,  "boi_ids_flat")
_SHM_REGISTRY["boi_vals_flat"] = ndarray_to_shm(vals_flat, "boi_vals_flat")
_SHM_REGISTRY["boi_offsets"]   = ndarray_to_shm(offsets,   "boi_offsets")
_SHM_REGISTRY["boi_sig_all"]   = ndarray_to_shm(sig_all,   "boi_sig_all")

print(f"Shared memory setup: {time.time()-t0:.2f} sec")
print(f"  boi_ids_flat  : {ids_flat.nbytes/1e9:.2f} GB")
print(f"  boi_vals_flat : {vals_flat.nbytes/1e9:.2f} GB")
print(f"  boi_offsets   : {offsets.nbytes/1e6:.2f} MB")
print(f"  boi_sig_all   : {sig_all.nbytes/1e9:.2f} GB")

# Store shapes/dtypes for workers to reconstruct views
SHM_META = {
    "ids_flat":  {"shape": ids_flat.shape,  "dtype": ids_flat.dtype,  "name": "boi_ids_flat"},
    "vals_flat": {"shape": vals_flat.shape, "dtype": vals_flat.dtype, "name": "boi_vals_flat"},
    "offsets":   {"shape": offsets.shape,   "dtype": offsets.dtype,   "name": "boi_offsets"},
    "sig_all":   {"shape": sig_all.shape,   "dtype": sig_all.dtype,   "name": "boi_sig_all"},
}

print_memory_stats("After shared memory setup")

Moving arrays to shared memory...
Shared memory setup: 0.89 sec
  boi_ids_flat  : 0.98 GB
  boi_vals_flat : 0.98 GB
  boi_offsets   : 0.40 MB
  boi_sig_all   : 0.02 GB
[After shared memory setup] RSS=5.84 GB | VMS=29.58 GB | Avail=1956.69/2015.68 GB | Used=2.9%


# Banding Setup

Now we split each weighted MinHash signature into bands.

Purpose:
- generate bucket keys from contiguous chunks of the signature
- group polygons that collide in at least one band
- preserve the overall structure of the earlier BOI pipeline

Definitions:
- `L` = signature length
- `B` = number of bands
- `R` = rows per band
- must satisfy: `L = B * R`

For now, we keep the same style as the earlier notebook so comparisons remain fair.

In [9]:
# ================================
# Banding configuration
# ================================

# Keep this aligned with the earlier pipeline for easier comparison
B = 50
R = 2

assert L == B * R, f"L must equal B * R, but got L={L}, B={B}, R={R}"

print("=== Banding Parameters ===")
print(f"L : {L}")
print(f"B : {B}")
print(f"R : {R}")


# ================================
# Banding helper
# ================================

class Banding:
    """
    Simple helper to split signatures into B bands of size R.
    """

    def __init__(self, B, R):
        self.B = B
        self.R = R
        self.L = B * R

    def get_band(self, sig, band_idx):
        """
        Return one band as a tuple of R signature values.
        """
        start = band_idx * self.R
        end = start + self.R
        return tuple(sig[start:end])

    def get_all_bands(self, sig):
        """
        Return all bands for one signature.
        """
        return [self.get_band(sig, b) for b in range(self.B)]


banding = Banding(B=B, R=R)


# ================================
# Split signature matrix
# ================================

sig_data = sig_all[:DATA_END]
sig_query = sig_all[QUERY_START:QUERY_END]

print("\n=== Signature Split Diagnostics ===")
print(f"sig_data shape   : {sig_data.shape}")
print(f"sig_query shape  : {sig_query.shape}")

print_memory_stats("After banding setup")

=== Banding Parameters ===
L : 100
B : 50
R : 2

=== Signature Split Diagnostics ===
sig_data shape   : (40000, 100)
sig_query shape  : (10000, 100)
[After banding setup] RSS=5.84 GB | VMS=29.58 GB | Avail=1956.69/2015.68 GB | Used=2.9%


In [10]:
from collections import defaultdict
import numpy as np

def _secondary_split_members(members, sig_data, band_idx, R, max_bucket_size):
    """
    Split a large bucket using the next R signature values as a secondary key.
    sig_data is DB-only signatures.
    """
    if len(members) <= max_bucket_size:
        return [members]

    L = sig_data.shape[1]
    sec_start = ((band_idx + 1) * R) % max(1, L - R + 1)
    sec_end = min(sec_start + R, L)

    groups = defaultdict(list)
    for db_pid in members:
        sec_key = tuple(sig_data[db_pid, sec_start:sec_end].tolist())
        groups[sec_key].append(db_pid)

    out = list(groups.values())

    # fallback if split is useless
    if len(out) == 1:
        out = []
        for i in range(0, len(members), max_bucket_size):
            out.append(members[i:i + max_bucket_size])

    return out


def rebalance_part_map(part_map, sig_data, R, merge_small_threshold=20, split_large_threshold=5000):
    """
    1. Merge tiny buckets into per-band overflow bucket
    2. Split huge buckets using secondary signature slice
    """
    merged_map = {}
    overflow_by_band = defaultdict(list)

    # merge small buckets
    for (band_idx, band_key), members in part_map.items():
        members = list(members)
        if merge_small_threshold > 0 and len(members) < merge_small_threshold:
            overflow_by_band[band_idx].extend(members)
        else:
            merged_map[(band_idx, band_key)] = members

    for band_idx, overflow_members in overflow_by_band.items():
        if overflow_members:
            uniq = sorted(set(overflow_members))
            merged_map[(band_idx, "__OVERFLOW__")] = uniq

    # split large buckets
    final_map = {}
    for (band_idx, band_key), members in merged_map.items():
        members = sorted(set(members))

        if band_key != "__OVERFLOW__" and len(members) > split_large_threshold:
            sub_buckets = _secondary_split_members(
                members=members,
                sig_data=sig_data,
                band_idx=band_idx,
                R=R,
                max_bucket_size=split_large_threshold
            )
            for j, sub_members in enumerate(sub_buckets):
                final_map[(band_idx, band_key, "SPLIT", j)] = np.asarray(sorted(set(sub_members)), dtype=np.int32)
        else:
            final_map[(band_idx, band_key)] = np.asarray(members, dtype=np.int32)

    stats = {}
    sizes = np.array([len(v) for v in final_map.values()], dtype=np.int32)
    stats["total_buckets"] = len(final_map)
    stats["mean_bucket_size"] = float(sizes.mean()) if len(sizes) else 0.0
    stats["median_bucket_size"] = float(np.median(sizes)) if len(sizes) else 0.0
    stats["max_bucket_size"] = int(sizes.max()) if len(sizes) else 0
    stats["p90_bucket_size"] = float(np.percentile(sizes, 90)) if len(sizes) else 0.0
    stats["p99_bucket_size"] = float(np.percentile(sizes, 99)) if len(sizes) else 0.0

    return final_map, stats

# Build Band Partitions

This section builds the band-level partition map for the database polygons.

For each database polygon:
- take its weighted MinHash signature
- split it into `B` bands
- use each band tuple as a bucket key

Output:
- a mapping from `(band_id, band_key)` to the list of database polygon IDs that fall into that bucket

This restores the same broad BOI-style structure as the earlier notebook, but now using weighted MinHash signatures.

In [11]:
# ================================
# Build partition map from database signatures
# ================================

def build_partition_map(sig_data, banding):
    """
    Build the band partition map for database signatures.

    Parameters
    ----------
    sig_data : np.ndarray[int64] of shape (N_db, L)
        Signature matrix for database polygons only
    banding : Banding
        Banding helper object

    Returns
    -------
    part_map : dict
        Maps (band_idx, band_key) -> list of database polygon IDs
    bucket_sizes : list[int]
        Sizes of all buckets
    """
    part_map = defaultdict(list)

    for db_pid in tqdm(range(sig_data.shape[0]), desc="Building band partitions", unit="poly"):
        sig = sig_data[db_pid]

        for band_idx in range(banding.B):
            band_key = banding.get_band(sig, band_idx)
            part_map[(band_idx, band_key)].append(db_pid)

    bucket_sizes = [len(v) for v in part_map.values()]
    return part_map, bucket_sizes


print("[4] Building partition map from weighted MinHash signatures...")
t0 = time.time()

part_map_raw, bucket_sizes_raw = build_partition_map(sig_data=sig_data, banding=banding)

REB_MERGE_SMALL = 0
REB_SPLIT_LARGE = 3000

part_map, rebalance_stats = rebalance_part_map(
    part_map=part_map_raw,
    sig_data=sig_data,
    R=banding.R,
    merge_small_threshold=REB_MERGE_SMALL,
    split_large_threshold=REB_SPLIT_LARGE
)

bucket_sizes = [len(v) for v in part_map.values()]

t_part = time.time() - t0

print(f"\nPartition build time      : {t_part:.2f} sec")
print(f"Total unique buckets      : {len(part_map):,}")
print(f"Total bucket assignments  : {sum(bucket_sizes):,}")

print("\n=== Rebalance Settings ===")
print(f"REB_MERGE_SMALL          : {REB_MERGE_SMALL}")
print(f"REB_SPLIT_LARGE          : {REB_SPLIT_LARGE}")

print("\n=== Rebalanced Bucket Stats ===")
for k, v in rebalance_stats.items():
    print(f"{k:25s}: {v}")

bucket_sizes_arr = np.array(bucket_sizes, dtype=np.int32)

print("\n=== Bucket Size Diagnostics ===")
print(f"Mean bucket size          : {bucket_sizes_arr.mean():.2f}")
print(f"Median bucket size        : {np.median(bucket_sizes_arr):.2f}")
print(f"Min bucket size           : {bucket_sizes_arr.min()}")
print(f"Max bucket size           : {bucket_sizes_arr.max()}")
print(f"p90 bucket size           : {np.percentile(bucket_sizes_arr, 90):.2f}")
print(f"p99 bucket size           : {np.percentile(bucket_sizes_arr, 99):.2f}")

# A few useful counts
for thresh in [2, 5, 10, 50, 100, 500, 1000]:
    cnt = int(np.sum(bucket_sizes_arr >= thresh))
    print(f"Buckets with size >= {thresh:4d} : {cnt:,}")

# Show a few largest buckets
largest_items = sorted(part_map.items(), key=lambda kv: len(kv[1]), reverse=True)[:5]

print_memory_stats("After partition map build")

[4] Building partition map from weighted MinHash signatures...


Building band partitions: 100%|██████████| 40000/40000 [00:02<00:00, 13362.35poly/s]



Partition build time      : 3.79 sec
Total unique buckets      : 73,956
Total bucket assignments  : 2,000,000

=== Rebalance Settings ===
REB_MERGE_SMALL          : 0
REB_SPLIT_LARGE          : 3000

=== Rebalanced Bucket Stats ===
total_buckets            : 73956
mean_bucket_size         : 27.043106712099085
median_bucket_size       : 2.0
max_bucket_size          : 6219
p90_bucket_size          : 24.0
p99_bucket_size          : 599.4499999999971

=== Bucket Size Diagnostics ===
Mean bucket size          : 27.04
Median bucket size        : 2.00
Min bucket size           : 1
Max bucket size           : 6219
p90 bucket size           : 24.00
p99 bucket size           : 599.45
Buckets with size >=    2 : 37,005
Buckets with size >=    5 : 19,518
Buckets with size >=   10 : 12,755
Buckets with size >=   50 : 4,803
Buckets with size >=  100 : 3,131
Buckets with size >=  500 : 880
Buckets with size >= 1000 : 408
[After partition map build] RSS=5.92 GB | VMS=29.73 GB | Avail=1956.62/2015.68 

# Weighted Jaccard Helper for Reranking

This section defines the weighted Jaccard similarity used during the final reranking step.

In the BOI pipeline:
- weighted MinHash and banding generate candidates
- weighted Jaccard is then used to score and rerank those candidates exactly

This helper is part of the main retrieval pipeline.

In [12]:
# ================================
# Weighted Jaccard helper for reranking
# ================================

from numba import njit

@njit(cache=True)
def _weighted_jaccard_sparse_numba(ids_a, vals_a, ids_b, vals_b):
    if ids_a.size == 0 and ids_b.size == 0:
        return 1.0

    if ids_a.size == 0 or ids_b.size == 0:
        return 0.0

    i = 0
    j = 0
    num = 0.0
    den = 0.0

    while i < ids_a.size and j < ids_b.size:
        ida = ids_a[i]
        idb = ids_b[j]

        if ida == idb:
            va = vals_a[i]
            vb = vals_b[j]

            if va < vb:
                num += va
                den += vb
            else:
                num += vb
                den += va

            i += 1
            j += 1

        elif ida < idb:
            den += vals_a[i]
            i += 1

        else:
            den += vals_b[j]
            j += 1

    while i < ids_a.size:
        den += vals_a[i]
        i += 1

    while j < ids_b.size:
        den += vals_b[j]
        j += 1

    if den <= 0.0:
        return 1.0

    return num / den


def weighted_jaccard_from_tuples(enc_a, enc_b):
    ids_a, vals_a = enc_a
    ids_b, vals_b = enc_b
    return _weighted_jaccard_sparse_numba(ids_a, vals_a, ids_b, vals_b)

# Candidate Retrieval from Band Partitions

This section uses the weighted MinHash band partitions to generate candidates for each query.

For a given query:
- split its signature into `B` bands
- look up matching database buckets
- collect all candidate database polygon IDs
- deduplicate them
- rerank candidates using exact weighted Jaccard

This gives us a clean approximate-retrieval pipeline before adding any ANN backend.

In [13]:
# ================================
# Candidate retrieval + exact rerank
# ================================

def get_candidates_from_bands(query_sig, part_map, banding):
    """
    Retrieve candidate database polygon IDs by probing all query bands.

    Parameters
    ----------
    query_sig : np.ndarray[int64] of shape (L,)
        Weighted MinHash signature for one query
    part_map : dict
        Maps (band_idx, band_key) -> list of database polygon IDs
    banding : Banding
        Banding helper

    Returns
    -------
    candidates : list[int]
        Deduplicated candidate database IDs
    stats : dict
        Basic diagnostics for the query
    """
    candidate_set = set()
    buckets_hit = 0
    postings_scanned = 0
    bucket_sizes = []

    for band_idx in range(banding.B):
        band_key = banding.get_band(query_sig, band_idx)
        probe_keys = []

        exact_key = (band_idx, band_key)
        if exact_key in part_map:
            probe_keys.append(exact_key)

        overflow_key = (band_idx, "__OVERFLOW__")
        if overflow_key in part_map:
            probe_keys.append(overflow_key)

        split_prefix = (band_idx, band_key, "SPLIT")
        for k in part_map.keys():
            if len(k) == 4 and k[:3] == split_prefix:
                probe_keys.append(k)

        if not probe_keys:
            continue

        for pk in probe_keys:
            bucket = part_map[pk]
            buckets_hit += 1
            postings_scanned += len(bucket)
            bucket_sizes.append(len(bucket))

            for db_pid in bucket:
                candidate_set.add(int(db_pid))

    stats = {
        "buckets_hit": buckets_hit,
        "postings_scanned": postings_scanned,
        "num_candidates": len(candidate_set),
        "mean_hit_bucket_size": (float(np.mean(bucket_sizes)) if bucket_sizes else 0.0),
        "max_hit_bucket_size": (max(bucket_sizes) if bucket_sizes else 0)
    }

    return list(candidate_set), stats


def rerank_candidates_weighted_jaccard(query_enc, candidate_ids, database_encodings, topk=500):
    """
    Exact rerank of candidates using weighted Jaccard similarity.

    Returns
    -------
    results : list[(db_pid, sim)]
        Sorted descending by similarity
    """
    scored = []

    for db_pid in candidate_ids:
        sim = weighted_jaccard_from_tuples(query_enc, database_encodings[db_pid])
        scored.append((db_pid, sim))

    scored.sort(key=lambda x: (-x[1], x[0]))
    return scored[:topk]


def query_weighted_banding_one(qid, encodings, sig_all, part_map, banding, data_end, topk=500):
    """
    Run one approximate query:
    1) get candidates from band collisions
    2) rerank exactly with weighted Jaccard
    """
    query_sig = sig_all[qid]
    query_enc = encodings[qid]
    database_encodings = encodings[:data_end]

    candidate_ids, stats = get_candidates_from_bands(
        query_sig=query_sig,
        part_map=part_map,
        banding=banding
    )

    reranked = rerank_candidates_weighted_jaccard(
        query_enc=query_enc,
        candidate_ids=candidate_ids,
        database_encodings=database_encodings,
        topk=topk
    )

    return reranked, stats

print_memory_stats("After single-query smoke test")

[After single-query smoke test] RSS=5.92 GB | VMS=29.73 GB | Avail=1956.62/2015.68 GB | Used=2.9%


# Build Bucket Indexes

This section builds the bag of indexes (BOI) structure.

Design:
- only sufficiently large buckets get a local HNSW index
- smaller buckets are marked exact-only and searched directly at query time

Implementation detail:
- we first split buckets into large vs small
- then build indexes only for the large buckets

This avoids misleading progress estimates and keeps the multi-index stage practical.

In [14]:
# ================================
# Build bucket-local indexes in parallel (BOI core, PROCESSPOOL + reload)
# ================================

import os
import time
import shutil
import tempfile
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor, as_completed

import nmslib

# -------------------------
# Parallel build settings
# -------------------------
BUCKET_INDEX_MIN_SIZE = 1000
BUILD_WORKERS = 36
NUM_THREADS = 4

LOCAL_HNSW_M = 16
LOCAL_HNSW_EFC = 150
LOCAL_HNSW_EFS = 400

# Keep variable name for summary compatibility
BUCKET_INDEX_DIR = "IN_MEMORY_ONLY"

# Nested LSH parameters for large buckets — keep small to control memory and index build time
nested_L2 = 40
nested_R2 = 2

# IVF parameters for large buckets — keep small to control memory and index build time
IVF_NLIST  = 16   # number of IVF clusters per bucket
IVF_NPROBE = 4   # clusters to probe at query time.

RP_K        = 12   # number of random projection bits (hash code length)
RP_N_TABLES = 10   # number of independent hash tables

# Persistent index cache — keyed to dataset + HNSW config so changing
# any parameter automatically invalidates and rebuilds
_INDEX_CACHE_KEY = (
    f"L{L}_B{banding.B}_R{banding.R}"
    f"_M{LOCAL_HNSW_M}_efc{LOCAL_HNSW_EFC}"
    f"_minSz{BUCKET_INDEX_MIN_SIZE}"
    f"_rebSplit{REB_SPLIT_LARGE}"
    f"_{INDEX_TYPE}"
    f"_nL{nested_L2}_nR{nested_R2}"
    f"_ivfNl{IVF_NLIST}_ivfNp{IVF_NPROBE}"
    f"_sigvec"
)
BUCKET_INDEX_CACHE_DIR = os.path.join(
    "/raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache",
    _dataset_key,
    f"bucket_indexes_{_INDEX_CACHE_KEY}"
)
print(f"BUCKET_INDEX_CACHE_DIR : {BUCKET_INDEX_CACHE_DIR}")

print("=== Parallel Bucket Index Build Parameters ===")
print(f"BUCKET_INDEX_MIN_SIZE  : {BUCKET_INDEX_MIN_SIZE}")
print(f"BUILD_WORKERS          : {BUILD_WORKERS}")
print(f"NUM_THREADS            : {NUM_THREADS}")
print(f"LOCAL_HNSW_M           : {LOCAL_HNSW_M}")
print(f"LOCAL_HNSW_EFC         : {LOCAL_HNSW_EFC}")
print(f"LOCAL_HNSW_EFS         : {LOCAL_HNSW_EFS}")
print(f"BUCKET_INDEX_DIR       : {BUCKET_INDEX_DIR}")
print(f"INDEX_TYPE             : {INDEX_TYPE}")


MAX_BUCKET_INDEX_SIZE = 2000

def split_buckets_by_size(part_map, min_bucket_size):
    """
    Split buckets into:
    - large buckets: build local HNSW index
    - small buckets: exact-only at query time
    """
    large = []
    small = []

    for bucket_key, members in part_map.items():
        member_ids = np.asarray(members, dtype=np.int32)
        size = int(member_ids.size)
        item = (bucket_key, member_ids, size)

        if size >= min_bucket_size:
            large.append(item)
        else:
            small.append(item)

    large.sort(key=lambda x: x[2], reverse=True)
    small.sort(key=lambda x: x[2], reverse=True)

    return large, small


# -------------------------
# Build-worker globals
# -------------------------
_BUCKET_BUILD_GLOBALS = {}


def _init_bucket_build_worker(shm_meta,
                               local_hnsw_m,
                               local_hnsw_efc,
                               local_hnsw_efs,
                               num_threads,
                               tmp_index_dir):
    from multiprocessing.shared_memory import SharedMemory
    global _BUCKET_BUILD_GLOBALS

    # Attach to shared sig_all — 100-dim WMH signatures, much smaller than x_dense
    meta = shm_meta["sig_all"]
    _shm_sig = SharedMemory(create=False, name=meta["name"])
    sig_all_shm = np.ndarray(meta["shape"], dtype=meta["dtype"], buffer=_shm_sig.buf)

    _BUCKET_BUILD_GLOBALS = {
        "sig_all": sig_all_shm,
        "_shm_sig": _shm_sig,
        "local_hnsw_m": local_hnsw_m,
        "local_hnsw_efc": local_hnsw_efc,
        "local_hnsw_efs": local_hnsw_efs,
        "num_threads": num_threads,
        "tmp_index_dir": tmp_index_dir,
        "nested_L2": nested_L2,
        "nested_R2": nested_R2,
        "ivf_nprobe": IVF_NPROBE,
        "ivf_nlist":  IVF_NLIST,
        "ivf_nprobe": IVF_NPROBE,
        "rp_k":        RP_K,
        "rp_n_tables": RP_N_TABLES,
    }

def _build_one_bucket_nested_lsh(args):
    """
    Build a nested LSH index for one large bucket.
    Applies a second round of MinHash banding on the bucket's signatures.
    Returns a dict of inner_band_key -> member_ids (as np.int32 arrays).
    """
    bucket_key, member_ids = args
    G = _BUCKET_BUILD_GLOBALS

    X_bucket = G["sig_all"][member_ids]   # shape: (bucket_size, 100)
    L2 = G["nested_L2"]
    R2 = G["nested_R2"]
    B2 = L2 // R2

    inner_map = {}
    for local_idx in range(len(member_ids)):
        sig = X_bucket[local_idx]
        for b in range(B2):
            band_key = tuple(sig[b * R2 : b * R2 + R2].tolist())
            k = (b, band_key)
            if k not in inner_map:
                inner_map[k] = []
            inner_map[k].append(local_idx)

    # Convert to numpy arrays
    inner_map_np = {
        k: np.asarray(v, dtype=np.int32)
        for k, v in inner_map.items()
    }

    return {
        "bucket_key": bucket_key,
        "member_ids": member_ids,
        "inner_map":  inner_map_np,
        "use_index":  True,
        "index_type": "nested_lsh",
        "worker_elapsed_sec": 0.0,
    }

def _build_one_bucket_ivf_flat(args):
    """
    Build a FAISS IVFFlat index for one large bucket.
    Uses 100-dim MinHash signatures as input vectors.
    """
    import faiss

    bucket_key, member_ids = args
    t0_worker = time.time()
    G = _BUCKET_BUILD_GLOBALS

    X_bucket = G["sig_all"][member_ids].astype(np.float32)  # (bucket_size, 100)
    bucket_size = len(member_ids)
    dim = X_bucket.shape[1]

    # nlist must be <= bucket_size
    nlist = min(G["ivf_nlist"], bucket_size)

    quantizer = faiss.IndexFlatL2(dim)
    idx = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_L2)
    idx.train(X_bucket)
    idx.add(X_bucket)
    idx.nprobe = G["ivf_nprobe"]

    return {
        "bucket_key":         bucket_key,
        "member_ids":         member_ids,
        "index":              idx,
        "size":               int(bucket_size),
        "use_index":          True,
        "index_type":         "ivf_flat",
        "worker_elapsed_sec": float(time.time() - t0_worker),
    }

def _build_one_bucket_rp_lsh(args):
    """
    Build a Random Projection LSH index for one large bucket.
    Projects 100-dim signatures onto K random hyperplanes → K-bit binary code.
    Builds N independent hash tables for better recall.
    GPU-friendly: core operation is matrix multiply (dot product + sign).
    """
    bucket_key, member_ids = args
    t0_worker = time.time()
    G = _BUCKET_BUILD_GLOBALS

    X_bucket = G["sig_all"][member_ids].astype(np.float32)  # (bucket_size, 100)
    bucket_size = len(member_ids)
    dim = X_bucket.shape[1]
    K = G["rp_k"]
    N = G["rp_n_tables"]

    # Generate N sets of K random hyperplanes — fixed seed for reproducibility
    rng = np.random.RandomState(seed=42)
    hyperplanes = rng.randn(N, K, dim).astype(np.float32)  # (N, K, dim)

    # Compute binary codes for all members across all tables
    # projections[t] shape: (bucket_size, K)
    hash_tables = []
    for t in range(N):
        projections = X_bucket @ hyperplanes[t].T   # (bucket_size, K)
        binary_codes = (projections > 0).astype(np.uint8)  # (bucket_size, K)

        # Build hash table: code_tuple -> list of local_ids
        table = {}
        for local_idx in range(bucket_size):
            code = tuple(binary_codes[local_idx].tolist())
            if code not in table:
                table[code] = []
            table[code].append(local_idx)

        hash_tables.append(table)

    return {
        "bucket_key":         bucket_key,
        "member_ids":         member_ids,
        "size":               int(bucket_size),
        "use_index":          True,
        "index_type":         "rp_lsh",
        "hash_tables":        hash_tables,
        "hyperplanes":        hyperplanes,
        "worker_elapsed_sec": float(time.time() - t0_worker),
    }

def _build_one_bucket_index_worker(args):
    """
    Worker function:
    - receives one large bucket
    - builds local NMSLIB WeightedJaccard HNSW
    - saves it to a temp path
    - returns metadata only
    """
    import os
    import uuid
    import nmslib

    bucket_key, member_ids = args
    t0_worker = time.time()
    G = _BUCKET_BUILD_GLOBALS

    X_bucket = G["sig_all"][member_ids].astype(np.float32)

    idx = nmslib.init(space='WeightedJaccard', method='hnsw')
    idx.addDataPointBatch(X_bucket)

    idx.createIndex(
        {
            'M': G["local_hnsw_m"],
            'efConstruction': G["local_hnsw_efc"],
            'indexThreadQty': G["num_threads"],
            'post': 1
        },
        print_progress=False
    )
    idx.setQueryTimeParams({'efSearch': G["local_hnsw_efs"]})

    safe_name = f"bucket_{uuid.uuid4().hex}"
    index_path = os.path.join(G["tmp_index_dir"], safe_name)
    idx.saveIndex(index_path, save_data=True)

    # drop worker-local reference
    idx = None
    t_worker_done = time.time()

    return {
        "bucket_key": bucket_key,
        "member_ids": member_ids,
        "size": int(member_ids.size),
        "use_index": True,
        "index_path": index_path,
        "worker_elapsed_sec": float(t_worker_done - t0_worker),
    }


def _load_saved_weighted_index(index_path, efS):
    """
    Load one saved NMSLIB WeightedJaccard index into the parent process.
    """
    idx = nmslib.init(space='WeightedJaccard', method='hnsw')
    idx.loadIndex(index_path, load_data=True)
    idx.setQueryTimeParams({'efSearch': efS})
    return idx


def build_bucket_indexes_parallel(part_map,
                                  min_bucket_size=300,
                                  bucket_index_dir="IN_MEMORY_ONLY",
                                  build_workers=20,
                                  num_threads=8,
                                  local_hnsw_m=16,
                                  local_hnsw_efc=200,
                                  local_hnsw_efs=400):
    """
    Parallel BOI local-index builder (processpool version).

    Large buckets:
      - built in worker processes
      - saved to temp paths
      - reloaded in parent process as live indexes

    Small buckets:
      - stored as exact-only
      - searched directly at query time
    """
    t_build_0 = time.time()

    large_buckets, small_buckets = split_buckets_by_size(
        part_map=part_map,
        min_bucket_size=min_bucket_size
    )

    print("\n=== Bucket Split Summary ===")
    print(f"Total buckets             : {len(part_map):,}")
    print(f"Large buckets (indexed)   : {len(large_buckets):,}")
    print(f"Small buckets (exact-only): {len(small_buckets):,}")

    bucket_indexes = {}

    # Register small buckets first
    for bucket_key, member_ids, size in small_buckets:
        bucket_indexes[bucket_key] = {
            "member_ids": member_ids,
            "size": int(size),
            "use_index": False,
            "index": None,
            "index_path": None,
        }

    jobs = [(bucket_key, member_ids) for bucket_key, member_ids, _size in large_buckets]
    
    total_members_indexed = 0
    shm_root = "/dev/shm" if os.path.isdir("/dev/shm") else None
    tmp_index_dir = tempfile.mkdtemp(prefix="weighted_boi_hnsw_", dir=shm_root)

    try:
        ctx = mp.get_context("fork")

        # Select worker function based on INDEX_TYPE
        if INDEX_TYPE == "nested_lsh":
            _worker_fn = _build_one_bucket_nested_lsh
        elif INDEX_TYPE == "ivf_flat":
            _worker_fn = _build_one_bucket_ivf_flat
        elif INDEX_TYPE == "rp_lsh":
            _worker_fn = _build_one_bucket_rp_lsh
        else:
            _worker_fn = _build_one_bucket_index_worker

        with ProcessPoolExecutor(
            max_workers=build_workers,
            mp_context=ctx,
            initializer=_init_bucket_build_worker,
            initargs=(
                SHM_META,
                local_hnsw_m,
                local_hnsw_efc,
                local_hnsw_efs,
                num_threads,
                tmp_index_dir,
            )
        ) as ex:
            futures = [ex.submit(_worker_fn, job) for job in jobs]

            worker_times = []
            reload_times = []

            for fut in tqdm(as_completed(futures), total=len(futures), desc=f"Building local {INDEX_TYPE} indexes", unit="bucket"):
                res = fut.result()

                bucket_key = tuple(res["bucket_key"])
                member_ids = res["member_ids"]

                if INDEX_TYPE == "nested_lsh":
                    bucket_indexes[bucket_key] = {
                        "member_ids": member_ids,
                        "size":       int(member_ids.size),
                        "use_index":  True,
                        "index_type": "nested_lsh",
                        "inner_map":  res["inner_map"],
                        "index":      None,
                        "index_path": None,
                    }
                    worker_times.append(res["worker_elapsed_sec"])
                    reload_times.append(0.0)
                elif INDEX_TYPE == "ivf_flat":
                    bucket_indexes[bucket_key] = {
                        "member_ids": member_ids,
                        "size":       int(res["size"]),
                        "use_index":  True,
                        "index_type": "ivf_flat",
                        "inner_map":  None,
                        "index":      res["index"],
                        "index_path": None,
                    }
                    worker_times.append(res["worker_elapsed_sec"])
                    reload_times.append(0.0)

                elif INDEX_TYPE == "rp_lsh":
                    bucket_indexes[bucket_key] = {
                        "member_ids": member_ids,
                        "size":       int(res["size"]),
                        "use_index":  True,
                        "index_type": "rp_lsh",
                        "inner_map":  None,
                        "index":      None,
                        "index_path": None,
                        "hash_tables":  res["hash_tables"],
                        "hyperplanes":  res["hyperplanes"],
                    }
                    worker_times.append(res["worker_elapsed_sec"])
                    reload_times.append(0.0)
                    
                else:
                    t0_reload = time.time()
                    idx = _load_saved_weighted_index(res["index_path"], local_hnsw_efs)
                    reload_elapsed = time.time() - t0_reload

                    bucket_indexes[bucket_key] = {
                        "member_ids": member_ids,
                        "size":       int(res["size"]),
                        "use_index":  True,
                        "index_type": "hnsw",
                        "inner_map":  None,
                        "index":      idx,
                        "index_path": res["index_path"],
                    }
                    worker_times.append(res["worker_elapsed_sec"])
                    reload_times.append(reload_elapsed)

                total_members_indexed += int(member_ids.size)

    finally:
        shutil.rmtree(tmp_index_dir, ignore_errors=True)

    t_build = time.time() - t_build_0

    # ------------------------------------------------------------------
    # Build split_index: maps (band_idx, band_key, "SPLIT") -> sorted list
    # of full split-bucket keys. Built once here, used at every query to
    # replace the O(#all_buckets) scan in get_probe_bucket_keys().
    # ------------------------------------------------------------------
    split_index = {}
    for k in bucket_indexes:
        if len(k) == 4 and k[2] == "SPLIT":
            prefix = k[:3]                          # (band_idx, band_key, "SPLIT")
            if prefix not in split_index:
                split_index[prefix] = []
            split_index[prefix].append(k)
    for prefix in split_index:
        split_index[prefix].sort(key=lambda x: x[3])   # sort by sub-bucket index

    # ------------------------------------------------------------------
    # Precompute k_search per indexed bucket — removes the if/elif/else
    # branch from the hot query loop.
    # ------------------------------------------------------------------
    USE_HNSW_THRESHOLD = 1500
    for meta in bucket_indexes.values():
        if meta["use_index"]:
            sz = meta["size"]
            if sz < 200:
                meta["k_search"] = max(1, sz // 2)
            elif sz < 1000:
                meta["k_search"] = min(250, sz // 2)
            else:
                meta["k_search"] = min(400, sz // 2)
        else:
            meta["k_search"] = 0   # unused for exact buckets

    bucket_sizes = np.array([len(v) for v in part_map.values()], dtype=np.int32)

    build_stats = {
        "total_buckets": int(len(part_map)),
        "indexed_buckets": int(len(large_buckets)),
        "exact_only_buckets": int(len(small_buckets)),
        "total_members_indexed": int(total_members_indexed),
        "mean_bucket_size": float(bucket_sizes.mean()) if bucket_sizes.size > 0 else None,
        "median_bucket_size": float(np.median(bucket_sizes)) if bucket_sizes.size > 0 else None,
        "max_bucket_size": int(bucket_sizes.max()) if bucket_sizes.size > 0 else None,
        "build_time_sec": float(t_build),
        "bucket_index_dir": bucket_index_dir,
        "avg_worker_build_sec": float(np.mean(worker_times)) if len(worker_times) > 0 else None,
        "avg_reload_sec": float(np.mean(reload_times)) if len(reload_times) > 0 else None,
        "max_worker_build_sec": float(np.max(worker_times)) if len(worker_times) > 0 else None,
        "max_reload_sec": float(np.max(reload_times)) if len(reload_times) > 0 else None,
    }

    return bucket_indexes, split_index, build_stats

    return bucket_indexes, build_stats


print("\n[BOI] Building bucket-local indexes in parallel...")
t0 = time.time()

BUCKET_INDEX_CACHE_DIR : /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/pk-real50k0.002/bucket_indexes_L100_B50_R2_M16_efc150_minSz1000_rebSplit3000_ivf_flat_nL40_nR2_ivfNl16_ivfNp4_sigvec
=== Parallel Bucket Index Build Parameters ===
BUCKET_INDEX_MIN_SIZE  : 1000
BUILD_WORKERS          : 36
NUM_THREADS            : 4
LOCAL_HNSW_M           : 16
LOCAL_HNSW_EFC         : 150
LOCAL_HNSW_EFS         : 400
BUCKET_INDEX_DIR       : IN_MEMORY_ONLY
INDEX_TYPE             : ivf_flat

[BOI] Building bucket-local indexes in parallel...


In [15]:
import pickle, hashlib

def save_bucket_index_cache(bucket_indexes, split_index, build_stats, cache_dir):
    """
    Save all HNSW indexes + metadata to a persistent cache directory.
    Each indexed bucket's nmslib index is saved as a separate .bin file.
    All metadata (member_ids, k_search, split_index, stats) saved as one pickle.
    """
    os.makedirs(cache_dir, exist_ok=True)

    # Save each HNSW index to a named file
    meta_map = {}
    for bucket_key, entry in tqdm(bucket_indexes.items(), desc="Saving bucket indexes"):
        key_str = str(bucket_key)
        key_hash = hashlib.md5(key_str.encode()).hexdigest()

        record = {
            "member_ids": entry["member_ids"],
            "size":       entry["size"],
            "use_index":  entry["use_index"],
            "k_search":   entry["k_search"],
            "index_path": None,
        }

        index_type = entry.get("index_type", "hnsw")
        record["index_type"] = index_type
        record["inner_map"]  = entry.get("inner_map", None)
        record["hash_tables"] = entry.get("hash_tables", None)
        record["hyperplanes"] = entry.get("hyperplanes", None)

        if entry["use_index"] and entry["index"] is not None:
            idx_path = os.path.join(cache_dir, f"idx_{key_hash}.bin")
            if index_type == "ivf_flat":
                import faiss
                faiss.write_index(entry["index"], idx_path)
            elif index_type in ("nested_lsh", "rp_lsh"):
                pass  # stored in record directly
            else:
                entry["index"].saveIndex(idx_path, save_data=True)
            record["index_path"] = idx_path

        meta_map[bucket_key] = record

    # Save metadata + split_index + stats
    payload = {
        "meta_map":    meta_map,
        "split_index": split_index,
        "build_stats": build_stats,
    }
    with open(os.path.join(cache_dir, "payload.pkl"), "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

    print(f"Saved {len(meta_map):,} bucket entries to: {cache_dir}")


def load_bucket_index_cache(cache_dir, efs):
    """
    Load all HNSW indexes + metadata from a persistent cache directory.
    Returns bucket_indexes, split_index, build_stats — same as build function.
    """
    payload_path = os.path.join(cache_dir, "payload.pkl")
    if not os.path.exists(payload_path):
        return None, None, None

    print(f"Loading bucket index cache from: {cache_dir}")
    with open(payload_path, "rb") as f:
        payload = pickle.load(f)

    meta_map    = payload["meta_map"]
    split_index = payload["split_index"]
    build_stats = payload["build_stats"]

    bucket_indexes = {}
    n_indexed = sum(1 for v in meta_map.values() if v["use_index"])

    for bucket_key, record in tqdm(meta_map.items(), desc="Loading HNSW indexes"):
        idx = None
        index_type = record.get("index_type", "hnsw")

        if record["use_index"] and record.get("index_path"):
            if index_type == "ivf_flat":
                import faiss
                idx = faiss.read_index(record["index_path"])
            elif index_type == "nested_lsh":
                idx = None  # inner_map loaded below
            else:
                idx = nmslib.init(space='WeightedJaccard', method='hnsw')
                idx.loadIndex(record["index_path"], load_data=True)
                idx.setQueryTimeParams({'efSearch': efs})

        bucket_indexes[bucket_key] = {
            "member_ids":  record["member_ids"],
            "size":        record["size"],
            "use_index":   record["use_index"],
            "index":       idx,
            "index_path":  record.get("index_path"),
            "index_type":  index_type,
            "inner_map":   record.get("inner_map", None),
            "hash_tables": record.get("hash_tables", None),
            "hyperplanes": record.get("hyperplanes", None),
            "k_search":    record["k_search"],
        }

    print(f"Loaded {len(bucket_indexes):,} buckets "
          f"({n_indexed:,} indexed, "
          f"{len(bucket_indexes)-n_indexed:,} exact-only)")
    return bucket_indexes, split_index, build_stats


# -------------------------------------------------------
# Cache-aware build — load if available, build otherwise
# -------------------------------------------------------
t0 = time.time()
_payload_exists = False

# _payload_exists = os.path.exists(
#     os.path.join(BUCKET_INDEX_CACHE_DIR, "payload.pkl")
# )

if _payload_exists:
    print(f"[Cache hit] Loading bucket indexes from: {BUCKET_INDEX_CACHE_DIR}")
    bucket_indexes, split_index, bucket_index_build_stats = load_bucket_index_cache(
        cache_dir=BUCKET_INDEX_CACHE_DIR,
        efs=LOCAL_HNSW_EFS,
    )
else:
    print(f"[Cache miss] Building bucket indexes fresh...")
    bucket_indexes, split_index, bucket_index_build_stats = build_bucket_indexes_parallel(
        part_map=part_map,
        min_bucket_size=BUCKET_INDEX_MIN_SIZE,
        bucket_index_dir=BUCKET_INDEX_DIR,
        build_workers=BUILD_WORKERS,
        num_threads=NUM_THREADS,
        local_hnsw_m=LOCAL_HNSW_M,
        local_hnsw_efc=LOCAL_HNSW_EFC,
        local_hnsw_efs=LOCAL_HNSW_EFS,
    )
    print(f"[Cache save] Saving bucket indexes to: {BUCKET_INDEX_CACHE_DIR}")
    save_bucket_index_cache(
        bucket_indexes=bucket_indexes,
        split_index=split_index,
        build_stats=bucket_index_build_stats,
        cache_dir=BUCKET_INDEX_CACHE_DIR,
    )

t_bucket_build = time.time() - t0

print(f"\n=== Parallel Bucket Index Build Summary ===")
print(f"Total buckets             : {bucket_index_build_stats['total_buckets']:,}")
print(f"Indexed buckets           : {bucket_index_build_stats['indexed_buckets']:,}")
print(f"Exact-only buckets        : {bucket_index_build_stats['exact_only_buckets']:,}")
print(f"Total members indexed     : {bucket_index_build_stats['total_members_indexed']:,}")
print(f"Mean bucket size          : {bucket_index_build_stats['mean_bucket_size']:.2f}")
print(f"Median bucket size        : {bucket_index_build_stats['median_bucket_size']:.2f}")
print(f"Max bucket size           : {bucket_index_build_stats['max_bucket_size']:,}")
print(f"Build time                : {bucket_index_build_stats['build_time_sec']:.2f} sec ({bucket_index_build_stats['build_time_sec']/60:.2f} min)")
print(f"Index directory           : {bucket_index_build_stats['bucket_index_dir']}")
print(f"Avg worker build sec      : {bucket_index_build_stats['avg_worker_build_sec']:.4f}")
print(f"Avg reload sec            : {bucket_index_build_stats['avg_reload_sec']:.4f}")
print(f"Max worker build sec      : {bucket_index_build_stats['max_worker_build_sec']:.4f}")
print(f"Max reload sec            : {bucket_index_build_stats['max_reload_sec']:.4f}")

print_memory_stats("After parallel bucket-local index build")
import ctypes, gc
gc.collect()
ctypes.CDLL("libc.so.6").malloc_trim(0)
print_memory_stats("After malloc_trim post index build")

[Cache miss] Building bucket indexes fresh...

=== Bucket Split Summary ===
Total buckets             : 73,956
Large buckets (indexed)   : 408
Small buckets (exact-only): 73,548


Building local ivf_flat indexes: 100%|██████████| 408/408 [01:44<00:00,  3.92bucket/s]


[Cache save] Saving bucket indexes to: /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/pk-real50k0.002/bucket_indexes_L100_B50_R2_M16_efc150_minSz1000_rebSplit3000_ivf_flat_nL40_nR2_ivfNl16_ivfNp4_sigvec


Saving bucket indexes: 100%|██████████| 73956/73956 [00:00<00:00, 89556.46it/s] 


Saved 73,956 bucket entries to: /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/pk-real50k0.002/bucket_indexes_L100_B50_R2_M16_efc150_minSz1000_rebSplit3000_ivf_flat_nL40_nR2_ivfNl16_ivfNp4_sigvec

=== Parallel Bucket Index Build Summary ===
Total buckets             : 73,956
Indexed buckets           : 408
Exact-only buckets        : 73,548
Total members indexed     : 747,294
Mean bucket size          : 27.04
Median bucket size        : 2.00
Max bucket size           : 6,219
Build time                : 107.09 sec (1.78 min)
Index directory           : IN_MEMORY_ONLY
Avg worker build sec      : 8.8776
Avg reload sec            : 0.0000
Max worker build sec      : 12.0040
Max reload sec            : 0.0000
[After parallel bucket-local index build] RSS=6.30 GB | VMS=29.29 GB | Avail=1956.10/2015.68 GB | Used=3.0%
[After malloc_trim post index build] RSS=6.27 GB | VMS=29.27 GB | Avail=1956.12/2015.68 GB | Used=3.0%


In [16]:
def get_probe_bucket_keys(bucket_indexes, split_index, band_idx, band_key, max_split_probes=2):
    """
    O(1) probe-key lookup.

    split_index: dict mapping (band_idx, band_key, "SPLIT") -> sorted list of full 4-tuple keys.
    Replaces the previous O(#all_buckets) full-dict scan.
    """
    keys = []

    exact_key = (band_idx, band_key)
    if exact_key in bucket_indexes:
        keys.append(exact_key)

    prefix = (band_idx, band_key, "SPLIT")
    split_keys = split_index.get(prefix)
    if split_keys:
        keys.extend(split_keys[:max_split_probes])

    return keys

In [17]:
# ================================
# Fast parallel BOI evaluation wrapper (processpool, inherited in-memory indexes)
# ================================
import heapq
import math
import multiprocessing as mp
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed

_QBOI_GLOBALS = {}

def query_boi_weighted_fast_inmem(query_qid,
                                  topK=500,
                                  alpha_mult=0.10,
                                  epsilon=2000,
                                  efS=400,
                                  hnsw_k_mult=1,
                                  return_timing=False):
    G = _QBOI_GLOBALS

    t_query_start = time.time()

    B = G["banding"].B
    alpha = B * alpha_mult

    query_sig = G["sig_all"][query_qid]

    # Reconstruct sparse encoding from flat arrays (shared memory, zero-copy)
    q_start = int(G["offsets"][query_qid])
    q_end   = int(G["offsets"][query_qid + 1])
    query_ids_w  = G["ids_flat"][q_start:q_end]
    query_vals_w = G["vals_flat"][q_start:q_end]

    # 100-dim signature vector for bucket-local HNSW
    # Must match what was used at index build time
    query_vec = G["sig_all"][query_qid]

    scoreboard = np.zeros(G["data_end"], dtype=np.float32)
    scoreboard_touched = []

    buckets_probed = 0
    postings_scanned = 0
    hnsw_queries = 0

    indexed_buckets_hit = 0
    exact_buckets_hit = 0
    indexed_candidates_added = 0
    exact_candidates_added = 0

    t_probe_start = time.time()
    t_hnsw_total = 0.0
    t_exact_total = 0.0

    # ✅ Keep this (best from sweep)
    MAX_BANDS_TO_PROBE = 40

    # --------------------------------------------------
    # SIMPLE + EFFECTIVE: first 60 bands (no sorting)
    # --------------------------------------------------
    for b in range(min(B, MAX_BANDS_TO_PROBE)):
        band_key = G["banding"].get_band(query_sig, b)
        probe_keys = get_probe_bucket_keys(G["bucket_indexes"], G["split_index"], b, band_key)

        if not probe_keys:
            continue

        for bucket_key in probe_keys:
            meta = G["bucket_indexes"].get(bucket_key)
            if meta is None:
                continue

            member_ids = meta["member_ids"]
            if member_ids is None or len(member_ids) == 0:
                continue

            buckets_probed += 1
            w = 1.0
            bucket_size = int(member_ids.size)

            # --------------------------------------------------
            # existing logic stays the same below
            # --------------------------------------------------
            USE_HNSW_THRESHOLD = 1500

            if meta["use_index"] and bucket_size >= USE_HNSW_THRESHOLD:
                indexed_buckets_hit += 1
                t_hnsw_0 = time.time()

                if meta.get("index_type") == "nested_lsh":
                    # Nested LSH: hash query into inner bands, collect candidates
                    inner_map = meta["inner_map"]
                    nested_L2 = G.get("nested_L2", 40)
                    nested_R2 = G.get("nested_R2", 2)
                    nested_B2 = nested_L2 // nested_R2
                    candidate_local = set()

                    for ib in range(nested_B2):
                        inner_key = (ib, tuple(query_vec[ib * nested_R2 : ib * nested_R2 + nested_R2].tolist()))
                        if inner_key in inner_map:
                            for local_id in inner_map[inner_key]:
                                candidate_local.add(int(local_id))

                    local_ids = list(candidate_local)

                elif meta.get("index_type") == "ivf_flat":
                    # IVF-Flat: query the FAISS index
                    idx = meta["index"]
                    q = query_vec.astype(np.float32).reshape(1, -1)
                    k_search = meta["k_search"]
                    _, I = idx.search(q, k_search)
                    local_ids = [int(x) for x in I[0] if x >= 0]

                elif meta.get("index_type") == "rp_lsh":
                    # RP-LSH: project query into binary codes, probe hash tables
                    # Also probe Hamming distance 1 neighbors for better recall
                    hash_tables  = meta["hash_tables"]
                    hyperplanes  = meta["hyperplanes"]
                    q = query_vec.astype(np.float32)
                    K = hyperplanes.shape[1]
                    N = len(hash_tables)

                    candidate_local = set()
                    for t in range(N):
                        proj = hyperplanes[t] @ q          # (K,)
                        code = tuple((proj > 0).astype(np.uint8).tolist())

                        # Exact match
                        if code in hash_tables[t]:
                            for lid in hash_tables[t][code]:
                                candidate_local.add(int(lid))

                        # Hamming distance 1 probing — flip each bit
                        code_list = list(code)
                        for bit in range(K):
                            code_list[bit] = 1 - code_list[bit]
                            neighbor_code = tuple(code_list)
                            if neighbor_code in hash_tables[t]:
                                for lid in hash_tables[t][neighbor_code]:
                                    candidate_local.add(int(lid))
                            code_list[bit] = 1 - code_list[bit]  # flip back

                    local_ids = list(candidate_local)
                    
                else:
                    # HNSW path (existing)
                    hnsw_queries += 1
                    idx = meta["index"]
                    try:
                        local_ids, _ = idx.knnQuery(query_vec, k=meta["k_search"])
                    except RuntimeError:
                        local_ids = []

                indexed_candidates_added += len(local_ids)

                for local_id in local_ids:
                    gid = int(member_ids[int(local_id)])
                    if scoreboard[gid] == 0.0:
                        scoreboard_touched.append(gid)
                    scoreboard[gid] += w

                t_hnsw_total += (time.time() - t_hnsw_0)

            else:
                exact_buckets_hit += 1
                postings_scanned += bucket_size

                t_exact_0 = time.time()
                exact_candidates_added += bucket_size

                for gid in member_ids:
                    igid = int(gid)
                    if scoreboard[igid] == 0.0:
                        scoreboard_touched.append(igid)
                    scoreboard[igid] += w

                t_exact_total += (time.time() - t_exact_0)

    t_probe_end = time.time()

    # -------------------------
    # Prune
    # -------------------------
    t_prune_start = time.time()

    if alpha > 0:
        candidates = [(pid, scoreboard[pid]) for pid in scoreboard_touched if scoreboard[pid] >= alpha]
    else:
        candidates = []

    pool = candidates if candidates else [(pid, scoreboard[pid]) for pid in scoreboard_touched]
    if len(pool) > epsilon:
        _pool_ids = np.array([p[0] for p in pool], dtype=np.int32)
        _pool_scores = np.array([p[1] for p in pool], dtype=np.float32)
        _top_idx = np.argpartition(_pool_scores, -epsilon)[-epsilon:]
        top_eps = [(_pool_ids[i], _pool_scores[i]) for i in _top_idx]
    else:
        top_eps = pool

    # Reset touched entries for next query (avoids full array zero-fill)
    for pid in scoreboard_touched:
        scoreboard[pid] = 0.0
    scoreboard_touched.clear()

    t_prune_end = time.time()

    # -------------------------
    # Rerank
    # -------------------------
    t_rerank_start = time.time()

    reranked = []
    ids_flat_g  = G["ids_flat"]
    vals_flat_g = G["vals_flat"]
    offsets_g   = G["offsets"]

    for poly_id, _score in top_eps:
        s = int(offsets_g[poly_id])
        e = int(offsets_g[poly_id + 1])
        sim = _weighted_jaccard_sparse_numba(
            query_ids_w, query_vals_w,
            ids_flat_g[s:e], vals_flat_g[s:e]
        )
        reranked.append((poly_id, sim))

    reranked.sort(key=lambda x: (-x[1], x[0]))
    reranked_topk = reranked[:topK]

    t_rerank_end = time.time()
    t_query_end = time.time()

    log = {
        "buckets_probed": buckets_probed,
        "scoreboard_size": len(scoreboard_touched),
        "candidates_after_alpha": len(candidates),
        "postings_scanned": postings_scanned,
        "hnsw_queries": hnsw_queries,
        "hnsw_k_mult": hnsw_k_mult,
        "indexed_buckets_hit": indexed_buckets_hit,
        "exact_buckets_hit": exact_buckets_hit,
        "indexed_candidates_added": indexed_candidates_added,
        "exact_candidates_added": exact_candidates_added,
    }

    if return_timing:
        log["timing"] = {
            "probe_sec": t_probe_end - t_probe_start,
            "prune_sec": t_prune_end - t_prune_start,
            "rerank_sec": t_rerank_end - t_rerank_start,
            "total_sec": t_query_end - t_query_start,
            "hnsw_probe_sec": t_hnsw_total,
            "exact_probe_sec": t_exact_total,
        }

    return reranked_topk, log

def sparse_tuple_to_dense(enc, D, dtype=np.float32):
    ids, vals = enc
    x = np.zeros(D, dtype=dtype)
    if ids.size > 0:
        x[ids] = vals
    return x

def _init_query_boi_weighted_inmem_worker(banding, bucket_indexes, split_index,
                                          shm_meta, data_end, query_start, D):
    from multiprocessing.shared_memory import SharedMemory
    global _QBOI_GLOBALS

    # Attach shared arrays — no copy, no CoW
    _shm_ids   = SharedMemory(create=False, name=shm_meta["ids_flat"]["name"])
    _shm_vals  = SharedMemory(create=False, name=shm_meta["vals_flat"]["name"])
    _shm_off   = SharedMemory(create=False, name=shm_meta["offsets"]["name"])
    _shm_sig   = SharedMemory(create=False, name=shm_meta["sig_all"]["name"])

    ids_flat_w  = np.ndarray(shm_meta["ids_flat"]["shape"],  dtype=shm_meta["ids_flat"]["dtype"],  buffer=_shm_ids.buf)
    vals_flat_w = np.ndarray(shm_meta["vals_flat"]["shape"], dtype=shm_meta["vals_flat"]["dtype"], buffer=_shm_vals.buf)
    offsets_w   = np.ndarray(shm_meta["offsets"]["shape"],   dtype=shm_meta["offsets"]["dtype"],   buffer=_shm_off.buf)
    sig_all_w   = np.ndarray(shm_meta["sig_all"]["shape"],   dtype=shm_meta["sig_all"]["dtype"],   buffer=_shm_sig.buf)

    _QBOI_GLOBALS = {
        "banding":       banding,
        "bucket_indexes": bucket_indexes,
        "split_index":   split_index,
        "ids_flat":      ids_flat_w,
        "vals_flat":     vals_flat_w,
        "offsets":       offsets_w,
        "sig_all":       sig_all_w,
        "data_end":      data_end,
        "query_start":   query_start,
        "D":             D,
        "nested_L2": nested_L2,
        "nested_R2": nested_R2,
        # keep shm handles alive for duration of worker
        "_shm_ids":  _shm_ids,
        "_shm_vals": _shm_vals,
        "_shm_off":  _shm_off,
        "_shm_sig":  _shm_sig,
    }


def _query_boi_weighted_worker_chunk(args):
    qid_list, topK, alpha_mult, epsilon, efS, hnsw_k_mult, return_timing = args

    out = []

    chunk_t0 = time.time()

    for qid in qid_list:
        topk_res, log = query_boi_weighted_fast_inmem(
            query_qid=qid,
            topK=topK,
            alpha_mult=alpha_mult,
            epsilon=epsilon,
            efS=efS,
            hnsw_k_mult=hnsw_k_mult,
            return_timing=return_timing,
        )
        out.append((qid, topk_res, log))

    chunk_elapsed = time.time() - chunk_t0
    return out, chunk_elapsed


def _chunk_list(lst, n_chunks):
    if n_chunks <= 0:
        return [lst]
    chunk_sz = math.ceil(len(lst) / n_chunks)
    return [lst[i:i + chunk_sz] for i in range(0, len(lst), chunk_sz)]

if 'D' not in globals():
    max_id = -1
    for ids, vals in encodings:
        if ids.size > 0:
            local_max = int(ids.max())
            if local_max > max_id:
                max_id = local_max
    D = max_id + 1

print("D =", D)

# Recall computation
def compute_recall_at_k(pred_ids, gt_ids, k):
    if len(gt_ids) == 0:
        return None

    gt_k = gt_ids[:k]
    denom = min(k, len(gt_k))
    if denom == 0:
        return None

    pred_k = pred_ids[:k]
    hits = len(set(pred_k).intersection(gt_k))
    return hits / denom


def query_boi_weighted_parallel_processpool(query_ids,
                                            topK=500,
                                            alpha_mult=0.10,
                                            epsilon=2000,
                                            efS=400,
                                            hnsw_k_mult=1,
                                            num_workers=4,
                                            chunks_per_worker=8,
                                            return_timing=True,
                                            show_progress=True):
    """
    ProcessPool BOI evaluation using inherited in-memory bucket indexes.
    """
    n_queries = len(query_ids)

    if n_queries == 0:
        return {}, {}, {
            "num_queries": 0,
            "total_time_sec": 0.0,
            "qps": 0.0,
            "num_workers": num_workers,
            "chunks": 0,
        }

    total_chunks = max(1, num_workers * chunks_per_worker)
    qid_chunks = _chunk_list(query_ids, total_chunks)

    print("\n" + "=" * 80)
    print("Running query_boi_weighted_parallel_processpool")
    print("=" * 80)
    print(f"Queries            : {n_queries}")
    print(f"Workers            : {num_workers}")
    print(f"Chunks             : {len(qid_chunks)}")
    print(f"topK               : {topK}")
    print(f"alpha_mult         : {alpha_mult}")
    print(f"epsilon            : {epsilon}")
    print(f"efS                : {efS}")
    print(f"hnsw_k_mult        : {hnsw_k_mult}")
    print("=" * 80)

    t0 = time.time()
    ctx = mp.get_context("fork")

    with ProcessPoolExecutor(
        max_workers=num_workers,
        mp_context=ctx,
        initializer=_init_query_boi_weighted_inmem_worker,
        initargs=(
            banding,
            bucket_indexes,
            split_index,
            SHM_META,
            DATA_END,
            QUERY_START,
            D,
        )
    ) as ex:
        futures = [
            ex.submit(
                _query_boi_weighted_worker_chunk,
                (
                    chunk,
                    topK,
                    alpha_mult,
                    epsilon,
                    efS,
                    hnsw_k_mult,
                    return_timing,
                )
            )
            for chunk in qid_chunks
        ]

        chunk_results = []
        chunk_times = []

        iterator = as_completed(futures)

        if show_progress:
            iterator = tqdm(iterator, total=len(futures), desc="Parallel query chunks")

        for fut in iterator:
            chunk_out, chunk_elapsed = fut.result()
            chunk_results.append(chunk_out)
            chunk_times.append(chunk_elapsed)

    total_time = time.time() - t0

    results_dict = {}
    logs_dict = {}

    for chunk in chunk_results:
        for qid, topk_res, log in chunk:
            results_dict[qid] = topk_res
            logs_dict[qid] = log

    qps = n_queries / total_time if total_time > 0 else 0.0

    summary = {
        "num_queries": n_queries,
        "total_time_sec": total_time,
        "qps": qps,
        "num_workers": num_workers,
        "chunks": len(qid_chunks),
    }

    if return_timing:
        probe_sum = prune_sum = rerank_sum = total_sum = 0.0
        hnsw_probe_sum = exact_probe_sum = 0.0
        timed_queries = 0

        indexed_buckets_sum = 0
        exact_buckets_sum = 0
        indexed_candidates_sum = 0
        exact_candidates_sum = 0
        postings_scanned_sum = 0
        hnsw_queries_sum = 0

        for log in logs_dict.values():
            indexed_buckets_sum += log.get("indexed_buckets_hit", 0)
            exact_buckets_sum += log.get("exact_buckets_hit", 0)
            indexed_candidates_sum += log.get("indexed_candidates_added", 0)
            exact_candidates_sum += log.get("exact_candidates_added", 0)
            postings_scanned_sum += log.get("postings_scanned", 0)
            hnsw_queries_sum += log.get("hnsw_queries", 0)

            if "timing" in log:
                timed_queries += 1
                probe_sum += log["timing"]["probe_sec"]
                prune_sum += log["timing"]["prune_sec"]
                rerank_sum += log["timing"]["rerank_sec"]
                total_sum += log["timing"]["total_sec"]
                hnsw_probe_sum += log["timing"].get("hnsw_probe_sec", 0.0)
                exact_probe_sum += log["timing"].get("exact_probe_sec", 0.0)

        if timed_queries > 0:
            summary["avg_probe_sec"] = probe_sum / timed_queries
            summary["avg_prune_sec"] = prune_sum / timed_queries
            summary["avg_rerank_sec"] = rerank_sum / timed_queries
            summary["avg_query_total_sec"] = total_sum / timed_queries

            summary["avg_hnsw_probe_sec"] = hnsw_probe_sum / timed_queries
            summary["avg_exact_probe_sec"] = exact_probe_sum / timed_queries

            summary["avg_indexed_buckets_hit"] = indexed_buckets_sum / timed_queries
            summary["avg_exact_buckets_hit"] = exact_buckets_sum / timed_queries
            summary["avg_indexed_candidates_added"] = indexed_candidates_sum / timed_queries
            summary["avg_exact_candidates_added"] = exact_candidates_sum / timed_queries
            summary["avg_postings_scanned"] = postings_scanned_sum / timed_queries
            summary["avg_hnsw_queries"] = hnsw_queries_sum / timed_queries

        if len(chunk_times) > 0:
            summary["avg_chunk_time_sec"] = float(np.mean(chunk_times))
            summary["max_chunk_time_sec"] = float(np.max(chunk_times))
            summary["min_chunk_time_sec"] = float(np.min(chunk_times))

    print("\n=== Parallel Query Summary ===")
    print(f"Queries              : {summary['num_queries']}")
    print(f"Total time (sec)     : {summary['total_time_sec']:.2f}")
    print(f"QPS                  : {summary['qps']:.2f}")
    print(f"Workers              : {summary['num_workers']}")
    print(f"Chunks               : {summary['chunks']}")

    if "avg_query_total_sec" in summary:
        print(f"Avg probe sec/query  : {summary['avg_probe_sec']:.4f}")
        print(f"Avg prune sec/query  : {summary['avg_prune_sec']:.4f}")
        print(f"Avg rerank sec/query : {summary['avg_rerank_sec']:.4f}")
        print(f"Avg total sec/query  : {summary['avg_query_total_sec']:.4f}")
        
        print(f"Avg HNSW probe sec   : {summary['avg_hnsw_probe_sec']:.4f}")
        print(f"Avg exact probe sec  : {summary['avg_exact_probe_sec']:.4f}")

        print(f"Avg indexed buckets  : {summary['avg_indexed_buckets_hit']:.2f}")
        print(f"Avg exact buckets    : {summary['avg_exact_buckets_hit']:.2f}")
        print(f"Avg HNSW candidates  : {summary['avg_indexed_candidates_added']:.2f}")
        print(f"Avg exact candidates : {summary['avg_exact_candidates_added']:.2f}")
        print(f"Avg postings scanned : {summary['avg_postings_scanned']:.2f}")
        print(f"Avg HNSW queries     : {summary['avg_hnsw_queries']:.2f}")

        print(f"Avg chunk time sec   : {summary['avg_chunk_time_sec']:.4f}")
        print(f"Min chunk time sec   : {summary['min_chunk_time_sec']:.4f}")
        print(f"Max chunk time sec   : {summary['max_chunk_time_sec']:.4f}")

    return results_dict, logs_dict, summary

D = 18381


In [18]:
# ================================
# Full BOI evaluation using fast query logic
# ================================

# Numba warmup
for _enc in encodings[:10]:
    pass

_warm_a = encodings[0]
_warm_b = encodings[1]
_ = weighted_jaccard_from_tuples(_warm_a, _warm_b)
print("Numba weighted Jaccard warmup done.")

# Full GT-backed query set
boi_eval_qids = [
    qid for qid in gt_qids
    if (gt_offsets[gt_qid_map[qid] + 1] - gt_offsets[gt_qid_map[qid]]) > 0
]

import random
random.seed(42)
random.shuffle(boi_eval_qids)

print("boi_eval_qids size:", len(boi_eval_qids))
print("first 3:", boi_eval_qids[:3])
print("last 3:", boi_eval_qids[-3:])

# Fast-query parameters
FAST_TOPK = 500
FAST_ALPHA_MULT = 0.08
FAST_EPSILON = 4000
FAST_EFS = 800
FAST_HNSW_K_MULT = 1
QUERY_WORKERS = 36
CHUNKS_PER_WORKER = 8

Numba weighted Jaccard warmup done.
boi_eval_qids size: 9417
first 3: [np.int32(46227), np.int32(40442), np.int32(40710)]
last 3: [np.int32(44782), np.int32(40434), np.int32(41941)]


In [19]:
results_fast, logs_fast, summary_fast = query_boi_weighted_parallel_processpool(
    query_ids=boi_eval_qids[:QUERY_END],
    topK=FAST_TOPK,
    alpha_mult=FAST_ALPHA_MULT,
    epsilon=FAST_EPSILON,
    efS=FAST_EFS,
    hnsw_k_mult=FAST_HNSW_K_MULT,
    num_workers=QUERY_WORKERS,
    chunks_per_worker=CHUNKS_PER_WORKER,
    return_timing=True,
    show_progress=True,
)


rec10_list = []
rec50_list = []
rec500_list = []

example_rows = []

for idx, qid in enumerate(sorted(results_fast.keys())):
    pred_ids = [pid for pid, _sim in results_fast[qid]]
    _gi = gt_qid_map[qid]
    gt_ids = gt_neighbors[gt_offsets[_gi]:gt_offsets[_gi + 1]]

    r10 = compute_recall_at_k(pred_ids, gt_ids, 10)
    r50 = compute_recall_at_k(pred_ids, gt_ids, 50)
    r500 = compute_recall_at_k(pred_ids, gt_ids, 500)

    if r10 is not None:
        rec10_list.append(r10)
    if r50 is not None:
        rec50_list.append(r50)
    if r500 is not None:
        rec500_list.append(r500)

    if idx < 5:
        example_rows.append({
            "qid": qid,
            "pred_top10": pred_ids[:10],
            "gt_top10": gt_ids[:10],
            "r10": r10,
            "r50": r50,
            "r500": r500,
            "log": logs_fast[qid],
        })

boi_fast_summary = {
    "num_queries": len(results_fast),
    "topK": FAST_TOPK,
    "alpha_mult": FAST_ALPHA_MULT,
    "epsilon": FAST_EPSILON,
    "efS": FAST_EFS,
    "hnsw_k_mult": FAST_HNSW_K_MULT,
    "query_workers": QUERY_WORKERS,
    "chunks_per_worker": CHUNKS_PER_WORKER,

    "mean_recall@10": float(np.mean(rec10_list)) if rec10_list else None,
    "mean_recall@50": float(np.mean(rec50_list)) if rec50_list else None,
    "mean_recall@500": float(np.mean(rec500_list)) if rec500_list else None,

    "total_time_sec": summary_fast["total_time_sec"],
    "qps": summary_fast["qps"],

    "avg_probe_sec": summary_fast.get("avg_probe_sec"),
    "avg_prune_sec": summary_fast.get("avg_prune_sec"),
    "avg_rerank_sec": summary_fast.get("avg_rerank_sec"),
    "avg_query_total_sec": summary_fast.get("avg_query_total_sec"),

    # New timing breakdown
    "avg_hnsw_probe_sec": summary_fast.get("avg_hnsw_probe_sec"),
    "avg_exact_probe_sec": summary_fast.get("avg_exact_probe_sec"),

    # New indexed vs exact stats
    "avg_indexed_buckets_hit": summary_fast.get("avg_indexed_buckets_hit"),
    "avg_exact_buckets_hit": summary_fast.get("avg_exact_buckets_hit"),
    "avg_indexed_candidates_added": summary_fast.get("avg_indexed_candidates_added"),
    "avg_exact_candidates_added": summary_fast.get("avg_exact_candidates_added"),
    "avg_postings_scanned": summary_fast.get("avg_postings_scanned"),
    "avg_hnsw_queries": summary_fast.get("avg_hnsw_queries"),

    # Chunk timing
    "avg_chunk_time_sec": summary_fast.get("avg_chunk_time_sec"),
    "min_chunk_time_sec": summary_fast.get("min_chunk_time_sec"),
    "max_chunk_time_sec": summary_fast.get("max_chunk_time_sec"),
}

import pandas as pd

all_logs = []

for qid in results_fast:
    row = {"qid": qid}
    
    # add recall
    pred_ids = [pid for pid, _ in results_fast[qid]]
    _gi = gt_qid_map[qid]
    gt_ids = gt_neighbors[gt_offsets[_gi]:gt_offsets[_gi + 1]]

    row["r10"] = compute_recall_at_k(pred_ids, gt_ids, 10)
    row["r50"] = compute_recall_at_k(pred_ids, gt_ids, 50)
    row["r500"] = compute_recall_at_k(pred_ids, gt_ids, 500)

    # merge logs
    row.update(logs_fast[qid])
    
    all_logs.append(row)

df_logs = pd.DataFrame(all_logs)
df_logs.to_csv("boi_query_logs.csv", index=False)

print("Saved full query-level logs to boi_query_logs.csv")

def print_stats(name, arr):
    arr = np.array(arr)
    print(f"\n{name}")
    print(f"  mean : {np.mean(arr):.2f}")
    print(f"  p50  : {np.percentile(arr, 50):.2f}")
    print(f"  p90  : {np.percentile(arr, 90):.2f}")
    print(f"  p99  : {np.percentile(arr, 99):.2f}")
    print(f"  max  : {np.max(arr):.2f}")

print("\n=== Full Fast BOI Recall Summary ===")
print(f"Queries evaluated         : {boi_fast_summary['num_queries']}")
print(f"Mean Recall@10            : {boi_fast_summary['mean_recall@10']:.4f}")
print(f"Mean Recall@50            : {boi_fast_summary['mean_recall@50']:.4f}")
print(f"Mean Recall@500           : {boi_fast_summary['mean_recall@500']:.4f}")

print("\n=== Full Fast BOI Runtime Summary ===")
print(f"Total time (sec)          : {boi_fast_summary['total_time_sec']:.2f}")
print(f"QPS                       : {boi_fast_summary['qps']:.2f}")
print(f"Avg probe sec/query       : {boi_fast_summary['avg_probe_sec']:.4f}")
print(f"Avg prune sec/query       : {boi_fast_summary['avg_prune_sec']:.4f}")
print(f"Avg rerank sec/query      : {boi_fast_summary['avg_rerank_sec']:.4f}")
print(f"Avg total sec/query       : {boi_fast_summary['avg_query_total_sec']:.4f}")

print("\n=== HNSW vs Exact Breakdown ===")
print(f"Avg HNSW probe sec/query  : {boi_fast_summary.get('avg_hnsw_probe_sec', 0.0):.4f}")
print(f"Avg exact probe sec/query : {boi_fast_summary.get('avg_exact_probe_sec', 0.0):.4f}")
print(f"Avg indexed buckets hit   : {boi_fast_summary.get('avg_indexed_buckets_hit', 0.0):.2f}")
print(f"Avg exact buckets hit     : {boi_fast_summary.get('avg_exact_buckets_hit', 0.0):.2f}")
print(f"Avg HNSW candidates added : {boi_fast_summary.get('avg_indexed_candidates_added', 0.0):.2f}")
print(f"Avg exact candidates add. : {boi_fast_summary.get('avg_exact_candidates_added', 0.0):.2f}")
print(f"Avg postings scanned      : {boi_fast_summary.get('avg_postings_scanned', 0.0):.2f}")
print(f"Avg HNSW queries          : {boi_fast_summary.get('avg_hnsw_queries', 0.0):.2f}")

print("\n=== Parallel Chunk Timing ===")
print(f"Avg chunk time sec        : {boi_fast_summary.get('avg_chunk_time_sec', 0.0):.4f}")
print(f"Min chunk time sec        : {boi_fast_summary.get('min_chunk_time_sec', 0.0):.4f}")
print(f"Max chunk time sec        : {boi_fast_summary.get('max_chunk_time_sec', 0.0):.4f}")

print_memory_stats("After full fast BOI evaluation")


Running query_boi_weighted_parallel_processpool
Queries            : 9417
Workers            : 36
Chunks             : 286
topK               : 500
alpha_mult         : 0.08
epsilon            : 4000
efS                : 800
hnsw_k_mult        : 1


Parallel query chunks: 100%|██████████| 286/286 [00:26<00:00, 10.60it/s]



=== Parallel Query Summary ===
Queries              : 9417
Total time (sec)     : 31.08
QPS                  : 303.00
Workers              : 36
Chunks               : 286
Avg probe sec/query  : 0.0233
Avg prune sec/query  : 0.0032
Avg rerank sec/query : 0.0715
Avg total sec/query  : 0.0981
Avg HNSW probe sec   : 0.0111
Avg exact probe sec  : 0.0114
Avg indexed buckets  : 15.22
Avg exact buckets    : 38.06
Avg HNSW candidates  : 6017.21
Avg exact candidates : 20409.74
Avg postings scanned : 20409.74
Avg HNSW queries     : 0.00
Avg chunk time sec   : 3.2452
Min chunk time sec   : 0.7604
Max chunk time sec   : 12.2190
Saved full query-level logs to boi_query_logs.csv

=== Full Fast BOI Recall Summary ===
Queries evaluated         : 9417
Mean Recall@10            : 0.9939
Mean Recall@50            : 0.9933
Mean Recall@500           : 0.9596

=== Full Fast BOI Runtime Summary ===
Total time (sec)          : 31.08
QPS                       : 303.00
Avg probe sec/query       : 0.0233
Avg pru

In [20]:
# ================================
# Experiment Summary
# ================================
import platform
import psutil
import numpy as np

print("=" * 80)
print("BOI POLYGON SIMILARITY SEARCH — EXPERIMENT SUMMARY")
print("=" * 80)

# ---- Dataset ----
print("\n── DATASET ──────────────────────────────────────────")
print(f"  Encoding dir          : {ENCODING_DIR}")
print(f"  Total polygons        : {POLY_COUNT:,}")
print(f"  Database polygons     : {DATA_END:,}")
print(f"  Query polygons        : {QUERY_END - QUERY_START:,}")
print(f"  Feature dimension D   : {D:,}")
print(f"  Mean NNZ / polygon    : {np.mean([len(ids) for ids, vals in encodings]):.0f}")
print(f"  Median NNZ / polygon  : {np.median([len(ids) for ids, vals in encodings]):.0f}")
print(f"  Sig dtype             : {sig_all.dtype}")
print(f"  Sig shape             : {sig_all.shape}")

# ---- Pipeline config ----
print("\n── PIPELINE CONFIGURATION ───────────────────────────")
print(f"  Signature length L    : {L}")
print(f"  Bands B               : {banding.B}")
print(f"  Rows per band R       : {banding.R}")
print(f"  Max bands to probe    : {FAST_TOPK and 'see MAX_BANDS_TO_PROBE in query fn'}")
print(f"  Bucket index min size : {BUCKET_INDEX_MIN_SIZE}")
print(f"  Rebalance split large : {REB_SPLIT_LARGE}")
print(f"  HNSW M                : {LOCAL_HNSW_M}")
print(f"  HNSW efConstruction   : {LOCAL_HNSW_EFC}")
print(f"  HNSW efSearch         : {LOCAL_HNSW_EFS}")
print(f"  Index cache dir       : {BUCKET_INDEX_CACHE_DIR}")

# ---- Bucket index stats ----
print("\n── BUCKET INDEX ─────────────────────────────────────")
print(f"  Total buckets         : {bucket_index_build_stats['total_buckets']:,}")
print(f"  HNSW-indexed buckets  : {bucket_index_build_stats['indexed_buckets']:,}")
print(f"  Exact-only buckets    : {bucket_index_build_stats['exact_only_buckets']:,}")
print(f"  Total members indexed : {bucket_index_build_stats['total_members_indexed']:,}")
print(f"  Max bucket size       : {bucket_index_build_stats['max_bucket_size']:,}")
print(f"  Mean bucket size      : {bucket_index_build_stats['mean_bucket_size']:.2f}")
print(f"  Build time            : {bucket_index_build_stats['build_time_sec']:.1f} sec  ({bucket_index_build_stats['build_time_sec']/60:.2f} min)")

# ---- Query parameters ----
print("\n── QUERY PARAMETERS ─────────────────────────────────")
print(f"  topK                  : {FAST_TOPK}")
print(f"  alpha_mult            : {FAST_ALPHA_MULT}")
print(f"  epsilon               : {FAST_EPSILON}")
print(f"  efSearch              : {FAST_EFS}")
print(f"  Workers               : {QUERY_WORKERS}")
print(f"  Chunks per worker     : {CHUNKS_PER_WORKER}")

# ---- Recall results ----
print("\n── RECALL RESULTS ───────────────────────────────────")
print(f"  Queries evaluated     : {boi_fast_summary['num_queries']:,}")
print(f"  Mean Recall@10        : {boi_fast_summary['mean_recall@10']:.4f}")
print(f"  Mean Recall@50        : {boi_fast_summary['mean_recall@50']:.4f}")
print(f"  Mean Recall@500       : {boi_fast_summary['mean_recall@500']:.4f}")

# Per-query recall distributions
print(f"\n  Recall@10 distribution:")
r10 = np.array(rec10_list)
print(f"    p50={np.percentile(r10,50):.4f}  p90={np.percentile(r10,90):.4f}  p99={np.percentile(r10,99):.4f}  min={r10.min():.4f}")
r50 = np.array(rec50_list)
print(f"  Recall@50 distribution:")
print(f"    p50={np.percentile(r50,50):.4f}  p90={np.percentile(r50,90):.4f}  p99={np.percentile(r50,99):.4f}  min={r50.min():.4f}")
r500 = np.array(rec500_list)
print(f"  Recall@500 distribution:")
print(f"    p50={np.percentile(r500,50):.4f}  p90={np.percentile(r500,90):.4f}  p99={np.percentile(r500,99):.4f}  min={r500.min():.4f}")

# ---- Timing breakdown ----
print("\n── TIMING BREAKDOWN (per query) ─────────────────────")
print(f"  Total QPS             : {boi_fast_summary['qps']:.2f}")
print(f"  Avg total sec         : {boi_fast_summary['avg_query_total_sec']:.4f}")
print(f"  Avg probe sec         : {boi_fast_summary['avg_probe_sec']:.4f}  ({boi_fast_summary['avg_probe_sec']/boi_fast_summary['avg_query_total_sec']*100:.1f}%)")
print(f"    └─ HNSW probe       : {boi_fast_summary['avg_hnsw_probe_sec']:.4f}  ({boi_fast_summary['avg_hnsw_probe_sec']/boi_fast_summary['avg_query_total_sec']*100:.1f}%)")
print(f"    └─ Exact probe      : {boi_fast_summary['avg_exact_probe_sec']:.4f}  ({boi_fast_summary['avg_exact_probe_sec']/boi_fast_summary['avg_query_total_sec']*100:.1f}%)")
print(f"  Avg prune sec         : {boi_fast_summary['avg_prune_sec']:.4f}  ({boi_fast_summary['avg_prune_sec']/boi_fast_summary['avg_query_total_sec']*100:.1f}%)")
print(f"  Avg rerank sec        : {boi_fast_summary['avg_rerank_sec']:.4f}  ({boi_fast_summary['avg_rerank_sec']/boi_fast_summary['avg_query_total_sec']*100:.1f}%)")
print(f"\n  Avg HNSW queries/q    : {boi_fast_summary['avg_hnsw_queries']:.2f}")
print(f"  Avg indexed cands/q   : {boi_fast_summary['avg_indexed_candidates_added']:.0f}")
print(f"  Avg exact cands/q     : {boi_fast_summary['avg_exact_candidates_added']:.0f}")
print(f"  Avg postings scanned  : {boi_fast_summary['avg_postings_scanned']:.0f}")
print(f"\n  Chunk timing:")
print(f"    avg={boi_fast_summary['avg_chunk_time_sec']:.2f}s  min={boi_fast_summary['min_chunk_time_sec']:.2f}s  max={boi_fast_summary['max_chunk_time_sec']:.2f}s")
print(f"  Load imbalance ratio  : {boi_fast_summary['max_chunk_time_sec']/boi_fast_summary['min_chunk_time_sec']:.2f}x")

# ---- Memory ----
print("\n── MEMORY ───────────────────────────────────────────")
m = get_memory_stats()
print(f"  RSS                   : {m['rss_gb']:.2f} GB")
print(f"  System total          : {m['total_gb']:.2f} GB")
print(f"  System used           : {m['used_pct']:.1f}%")
shm_total = sum(np.prod(v['shape']) * np.dtype(v['dtype']).itemsize
                for v in SHM_META.values())
print(f"  Shared memory total   : {shm_total/1e9:.2f} GB")

# ---- Ground truth ----
print("\n── GROUND TRUTH ─────────────────────────────────────")
gt_lengths = (gt_offsets[1:] - gt_offsets[:-1]).astype(np.int32)
print(f"  GT queries            : {len(gt_qids):,}")
print(f"  GT min query ID       : {gt_qids[0]}")
print(f"  GT max query ID       : {gt_qids[-1]}")
print(f"  Mean GT list length   : {gt_lengths.mean():.1f}")
print(f"  Min GT list length    : {gt_lengths.min()}")
print(f"  Max GT list length    : {gt_lengths.max()}")

# ---- System ----
print("\n── SYSTEM ───────────────────────────────────────────")
print(f"  Python                : {platform.python_version()}")
print(f"  CPU cores             : {psutil.cpu_count(logical=True)}")
print(f"  RAM                   : {psutil.virtual_memory().total/1e9:.0f} GB")

print("\n" + "=" * 80)
print("END OF EXPERIMENT SUMMARY")
print("=" * 80)

BOI POLYGON SIMILARITY SEARCH — EXPERIMENT SUMMARY

── DATASET ──────────────────────────────────────────
  Encoding dir          : /raid/ruban/encodings/pk-real50k0.002
  Total polygons        : 50,000
  Database polygons     : 40,000
  Query polygons        : 10,000
  Feature dimension D   : 18,381
  Mean NNZ / polygon    : 4889
  Median NNZ / polygon  : 3958
  Sig dtype             : float32
  Sig shape             : (50000, 100)

── PIPELINE CONFIGURATION ───────────────────────────
  Signature length L    : 100
  Bands B               : 50
  Rows per band R       : 2
  Max bands to probe    : see MAX_BANDS_TO_PROBE in query fn
  Bucket index min size : 1000
  Rebalance split large : 3000
  HNSW M                : 16
  HNSW efConstruction   : 150
  HNSW efSearch         : 400
  Index cache dir       : /raid/ruban/boi_multi_index/Dataset_FloatingPoint/boi_cache/pk-real50k0.002/bucket_indexes_L100_B50_R2_M16_efc150_minSz1000_rebSplit3000_ivf_flat_nL40_nR2_ivfNl16_ivfNp4_sigvec

── BU